# PTC-SSL — full reproducible run (notebooks 01–04 + 06 in one Kaggle notebook)
**Settings:** Accelerator = GPU T4 x2, Internet = On. Use **Save Version → Save & Run All (Commit)**; it runs headless (about 4–6 h) and keeps every output.

Sections: 01 dataset acquisition · 02 profiling · 03 preprocessing · 04 windowing · 06 full experiments (5 seeds, ablations, external validation, statistics). Outputs: `results/` (tables, figures, configs, logs), `PTC_SSL_PAPER_RESULTS.zip`, `PTC_SSL_IEEE_ACCESS_RESEARCH_PACKAGE.zip`.

# 01 · Dataset acquisition and integrity verification
**Project:** Physiology-Guided Self-Supervised Temporal Representation Learning for Label-Efficient Early Sepsis Prediction (PTC-SSL)

This notebook obtains the **official PhysioNet/Computing in Cardiology Challenge 2019** training data
(https://physionet.org/content/challenge-2019/1.0.0/, CC BY 4.0) and verifies it before anything else is built.

**What it does**
1. Locates the data: an attached Kaggle input that keeps the official `training_setA/` and `training_setB/` folders, or a download.
2. Downloads from PhysioNet's open-data S3 mirror (`s3://physionet-open/challenge-2019/1.0.0/`) and checks every file's **MD5 against the provider's ETag**. If that fails it falls back to `physionet.org` (size check only).
3. Verifies structure: 41 columns in official order, hospital identity from the folder, patient counts (A = 20,336, B = 20,000), hourly `ICULOS`, binary and monotone `SepsisLabel`, constant demographics.
4. Writes one compact hourly table plus a patient table, a SHA-256 fingerprint of every file, the official scoring script and a verified reference list.

**Kaggle settings:** *Internet → On* (needs a phone-verified account), unless you attach the raw files as an input. CPU is enough. Expected run time: about 10–25 min (download) plus about 3 min (parsing).

**Outputs** (`/kaggle/working/`):
`artifacts/physionet2019_hourly.parquet`, `artifacts/patients.parquet`, `artifacts/file_fingerprints.csv`,
`artifacts/evaluate_sepsis_score_official.py`, `results/dataset_manifest.json`, `results/references.bib`, `results/tables/integrity_checks.csv`.

Raw `.psv` files are written to `/tmp`, so they are never part of the notebook output or the research packages.

In [ ]:
# ===== PTC-SSL common setup (identical in every notebook; do not edit per notebook) =====
import os, sys, json, time, math, glob, shutil, random, hashlib, logging, platform, warnings
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Kaggle paths (overridable through environment variables for local testing only)
WORK = Path(os.environ.get("PTCSSL_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("PTCSSL_INPUT", "/kaggle/input"))
ART = WORK / "artifacts"   # data artifacts consumed by later notebooks (may contain patient-level data)
RES = WORK / "results"     # metrics / tables / figures / configs (no raw patient data) -> paper package
LOGS = WORK / "logs"
for _d in (ART, RES, LOGS, RES / "figures", RES / "tables", RES / "configs"):
    _d.mkdir(parents=True, exist_ok=True)

GLOBAL_SEED = 2026

def set_seed(seed: int = GLOBAL_SEED):
    """Seed python, numpy and (if present) torch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed()

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
        fh = logging.FileHandler(LOGS / f"{name}.log", mode="a")
        fh.setFormatter(fmt)
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(fh)
        logger.addHandler(sh)
        logger.propagate = False
    return logger

try:
    display  # noqa: F821  (exists inside Jupyter/Kaggle)
except NameError:
    def display(x):
        print(x)

def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def md5_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    return path

def load_json(path):
    with open(path) as f:
        return json.load(f)

def save_table(df: pd.DataFrame, path):
    """Save as parquet (Kaggle has pyarrow); fall back to pickle if parquet is unavailable."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except ImportError:
        alt = path.with_suffix(".pkl")
        df.to_pickle(alt)
        return alt

def load_table(path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix == ".parquet" and path.exists():
        return pd.read_parquet(path)
    alt = path.with_suffix(".pkl")
    if alt.exists():
        return pd.read_pickle(alt)
    if path.exists():
        return pd.read_pickle(path)
    raise FileNotFoundError(path)

_ARTIFACT_CACHE = {}

def find_artifact(name: str, producer: str) -> Path:
    """Locate an artifact produced by an earlier notebook.
    Search order: this session's /kaggle/working/artifacts, then any attached input (/kaggle/input/**).
    Table artifacts may exist as .parquet or .pkl."""
    if name in _ARTIFACT_CACHE:
        return _ARTIFACT_CACHE[name]
    stem, suffix = os.path.splitext(name)
    names = [name] + ([stem + ".pkl"] if suffix == ".parquet" else [])
    for n in names:
        p = ART / n
        if p.exists():
            _ARTIFACT_CACHE[name] = p
            return p
    hits = []
    if INPUT.exists():
        for dirpath, _, files in os.walk(INPUT, followlinks=True):
            if Path(dirpath).name == "artifacts":          # never pick up copies under results/
                hits += [Path(dirpath) / n for n in names if n in files]
    if hits:
        hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        if len(hits) > 1:
            print(f"[find_artifact] {len(hits)} copies of {name} found; using newest: {hits[0]}")
        _ARTIFACT_CACHE[name] = hits[0]
        return hits[0]
    raise FileNotFoundError(
        f"Artifact '{name}' not found. Run notebook {producer} first, then attach its output to this "
        f"notebook (Kaggle: Add Input -> Your Work -> Notebooks -> {producer})."
    )

def env_info() -> dict:
    info = {
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
    for mod in ("scipy", "sklearn", "matplotlib", "torch", "lightgbm"):
        try:
            m = __import__(mod)
            info[mod] = getattr(m, "__version__", "unknown")
        except ImportError:
            info[mod] = "not installed"
    try:
        import torch
        info["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return info

class StageTimer:
    """Context manager that logs wall-clock time of a stage (also used for the compute-cost table)."""
    def __init__(self, name, logger=None):
        self.name, self.logger = name, logger
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        self.seconds = time.time() - self.t0
        msg = f"[time] {self.name}: {self.seconds:.1f}s"
        (self.logger.info if self.logger else print)(msg)

# PhysioNet 2019 schema (verified against the official data description, physionet.org/content/challenge-2019/1.0.0/)
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos", "Calcium",
        "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate", "Magnesium", "Phosphate",
        "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets"]
DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
DYNAMIC_VARS = VITALS + LABS                       # 34 time-varying clinical variables
EXPECTED_COLUMNS = DYNAMIC_VARS + DEMOGRAPHICS + ["SepsisLabel"]   # 41 columns, official order

# Official utility parameters (physionetchallenges/evaluation-2019, evaluate_sepsis_score.py)
UTILITY_PARAMS = dict(dt_early=-12, dt_optimal=-6, dt_late=3, max_u_tp=1.0, min_u_fn=-2.0, u_fp=-0.05, u_tn=0.0)
print(f"[setup] WORK={WORK}  INPUT={INPUT}  seed={GLOBAL_SEED}")

In [ ]:
NB = "01_dataset_acquisition"
log = get_logger(NB)
log.info("Environment: %s", json.dumps(env_info()))

CFG = {
    "dataset": "PhysioNet/CinC Challenge 2019 - Early Prediction of Sepsis from Clinical Data, v1.0.0",
    "landing_page": "https://physionet.org/content/challenge-2019/1.0.0/",
    "license": "Creative Commons Attribution 4.0 International Public License (CC BY 4.0)",
    "s3_bucket_https": "https://physionet-open.s3.amazonaws.com",
    "s3_prefix": "challenge-2019/1.0.0/training/",
    "physionet_files": "https://physionet.org/files/challenge-2019/1.0.0/training/",
    "sha256sums_url": "https://physionet.org/files/challenge-2019/1.0.0/SHA256SUMS.txt",
    "eval_script_url": "https://raw.githubusercontent.com/physionetchallenges/evaluation-2019/master/evaluate_sepsis_score.py",
    "hospital_dirs": {"A": "training_setA", "B": "training_setB"},
    "expected_patients": {"A": 20336, "B": 20000},
    "download_threads": 16,
    "http_timeout_s": 60,
    "max_retries": 5,
}
# Test fixtures only: override expected counts (never set this on Kaggle)
if os.environ.get("PTCSSL_EXPECTED_COUNTS"):
    CFG["expected_patients"] = json.loads(os.environ["PTCSSL_EXPECTED_COUNTS"])
    log.warning("EXPECTED COUNTS OVERRIDDEN FOR TESTING: %s", CFG["expected_patients"])

RAW = Path(os.environ.get("PTCSSL_RAW_DIR", "/tmp/physionet2019"))   # outside /kaggle/working on purpose
RAW.mkdir(parents=True, exist_ok=True)
assert len(EXPECTED_COLUMNS) == 41
print(json.dumps(CFG, indent=2))

## 1. Locate an existing copy (attached input or earlier download)
A source is accepted only if it keeps the official `training_setA` / `training_setB` folders.
Flat Kaggle mirrors that merge both hospitals are rejected, because hospital identity is needed for the A→B / B→A external validation.

In [ ]:
def scan_source(root: Path):
    """Return {hospital: sorted list of psv paths} if root contains official set folders, else None."""
    found = {}
    for hosp, dname in CFG["hospital_dirs"].items():
        dirs = [d for d in root.rglob(dname) if d.is_dir()] if root.exists() else []
        if not dirs:
            return None
        files = sorted(dirs[0].glob("*.psv"))
        if not files:
            return None
        found[hosp] = files
    return found

source_mode, source_root, files_by_hosp = None, None, None
candidates = []
if os.environ.get("PTCSSL_LOCAL_SOURCE"):
    candidates.append(("local", Path(os.environ["PTCSSL_LOCAL_SOURCE"])))
candidates.append(("previous_download", RAW))
if INPUT.exists():
    candidates += [("kaggle_input", d) for d in sorted(INPUT.iterdir()) if d.is_dir()]

for mode, root in candidates:
    try:
        res = scan_source(root)
    except Exception as e:
        log.warning("Could not scan %s: %s", root, e)
        res = None
    if res and all(len(res[h]) == CFG["expected_patients"][h] for h in res):
        source_mode, source_root, files_by_hosp = mode, root, res
        break
    if res:
        log.warning("Source %s is incomplete (%s files); ignoring.", root, {h: len(v) for h, v in res.items()})

print("Existing complete source:", source_mode, source_root)

## 2. Download if needed (S3 mirror with MD5 = ETag verification → physionet.org fallback)

In [ ]:
import re
import threading
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed

_tls = threading.local()

def _session():
    import requests
    if not hasattr(_tls, "s"):
        _tls.s = requests.Session()
        _tls.s.headers["User-Agent"] = "PTC-SSL-research-download/1.0"
    return _tls.s

def http_get(url, stream=False):
    if os.environ.get("PTCSSL_OFFLINE"):          # local test fixtures only
        raise ConnectionError("offline test mode")
    last = None
    for attempt in range(CFG["max_retries"]):
        try:
            r = _session().get(url, timeout=CFG["http_timeout_s"], stream=stream)
            if r.status_code == 200:
                return r
            last = RuntimeError(f"HTTP {r.status_code} for {url}")
            if r.status_code in (403, 404):
                break
        except Exception as e:   # network errors -> retry with backoff
            last = e
        time.sleep(min(2 ** attempt, 30))
    raise last

def list_s3(prefix):
    """List objects with key, size and ETag (MD5 for single-part uploads)."""
    ns = "{http://s3.amazonaws.com/doc/2006-03-01/}"
    out, token = [], None
    while True:
        url = f"{CFG['s3_bucket_https']}/?list-type=2&prefix={prefix}"
        if token:
            from urllib.parse import quote
            url += "&continuation-token=" + quote(token)
        root = ET.fromstring(http_get(url).content)
        for c in root.findall(f"{ns}Contents"):
            key = c.find(f"{ns}Key").text
            if key.endswith(".psv"):
                out.append({"key": key, "size": int(c.find(f"{ns}Size").text),
                            "etag": c.find(f"{ns}ETag").text.strip('"')})
        if (root.find(f"{ns}IsTruncated").text or "").lower() != "true":
            break
        token = root.find(f"{ns}NextContinuationToken").text
    return out

def list_physionet(dname):
    html = http_get(CFG["physionet_files"] + dname + "/").text
    names = sorted(set(re.findall(r'href="(p\d+\.psv)"', html)))
    return [{"key": dname + "/" + n, "size": None, "etag": None} for n in names]

def download_one(url, dest: Path, size=None, etag=None):
    if dest.exists() and (size is None or dest.stat().st_size == size):
        if etag is None or "-" in etag or md5_file(dest) == etag:
            return "skipped"
    tmp = dest.with_suffix(".part")
    data = http_get(url).content
    tmp.write_bytes(data)
    if size is not None and tmp.stat().st_size != size:
        raise IOError(f"size mismatch {dest.name}")
    if etag and "-" not in etag and md5_file(tmp) != etag:
        raise IOError(f"MD5/ETag mismatch {dest.name}")
    tmp.replace(dest)
    return "downloaded"

download_log = {}
if source_mode is None:
    with StageTimer("download", log):
        for hosp, dname in CFG["hospital_dirs"].items():
            target = RAW / "training" / dname
            target.mkdir(parents=True, exist_ok=True)
            try:
                entries = list_s3(CFG["s3_prefix"] + dname + "/")
                base, method = CFG["s3_bucket_https"] + "/", "s3_https_md5_etag"
                assert len(entries) > 0, "empty S3 listing"
            except Exception as e:
                log.warning("S3 listing failed for %s (%s); falling back to physionet.org", dname, e)
                entries = list_physionet(dname)
                base, method = CFG["physionet_files"], "physionet_org_size_unchecked"
            log.info("%s: %d files listed via %s", dname, len(entries), method)
            status, errors = {"downloaded": 0, "skipped": 0}, []
            with ThreadPoolExecutor(CFG["download_threads"]) as ex:
                futs = {}
                for e in entries:
                    fname = e["key"].split("/")[-1]
                    url = base + e["key"] if method.startswith("s3") else CFG["physionet_files"] + e["key"]
                    futs[ex.submit(download_one, url, target / fname, e["size"], e["etag"])] = fname
                for i, f in enumerate(as_completed(futs), 1):
                    try:
                        status[f.result()] += 1
                    except Exception as err:
                        errors.append((futs[f], str(err)))
                    if i % 5000 == 0:
                        log.info("  %s: %d/%d", dname, i, len(futs))
            # one retry pass for failures
            for fname, _ in list(errors):
                e = next(x for x in entries if x["key"].endswith("/" + fname))
                url = base + e["key"] if method.startswith("s3") else CFG["physionet_files"] + e["key"]
                download_one(url, target / fname, e["size"], e["etag"])
                errors = [x for x in errors if x[0] != fname]
            download_log[hosp] = {"method": method, "listed": len(entries), **status, "errors": errors,
                                  "etag_md5_verified": method.startswith("s3")}
            if errors:
                raise RuntimeError(f"{len(errors)} files failed for {dname}: {errors[:5]}")
            # keep listing (with provider checksums) for the integrity record
            pd.DataFrame(entries).to_csv(ART / f"provider_listing_{hosp}.csv", index=False)
    files_by_hosp = scan_source(RAW)
    source_mode, source_root = "downloaded", RAW
    if files_by_hosp is None:
        raise RuntimeError("Download finished but official folder structure not found. Check Internet is ON.")

print("Source:", source_mode, source_root)
print({h: len(v) for h, v in files_by_hosp.items()})
print(json.dumps(download_log, indent=2)[:2000])

## 3. File-level integrity: counts, uniqueness, SHA-256 fingerprints, provider checksums

In [ ]:
checks = []   # (check, status, detail)
def check(name, ok, detail=""):
    checks.append({"check": name, "status": "PASS" if ok else "FAIL", "detail": str(detail)})
    (log.info if ok else log.error)("%s %s %s", "PASS" if ok else "FAIL", name, detail)
    return ok

for h, exp in CFG["expected_patients"].items():
    check(f"patient_count_{h}", len(files_by_hosp[h]) == exp, f"found={len(files_by_hosp[h])}, expected={exp}")

ids = {h: [p.stem for p in v] for h, v in files_by_hosp.items()}
overlap = set(ids["A"]) & set(ids["B"])
check("no_patient_id_overlap_A_B", len(overlap) == 0, f"overlap={len(overlap)}")
check("unique_ids_within_A", len(set(ids["A"])) == len(ids["A"]))
check("unique_ids_within_B", len(set(ids["B"])) == len(ids["B"]))

with StageTimer("fingerprint", log):
    rows = []
    with ThreadPoolExecutor(16) as ex:
        futs = {ex.submit(lambda p: (sha256_file(p), md5_file(p), p.stat().st_size), p): (h, p)
                for h, v in files_by_hosp.items() for p in v}
        for f in as_completed(futs):
            h, p = futs[f]
            s, m, sz = f.result()
            rows.append({"hospital": h, "file": p.name, "bytes": sz, "sha256": s, "md5": m})
fp = pd.DataFrame(rows).sort_values(["hospital", "file"]).reset_index(drop=True)
fp.to_csv(ART / "file_fingerprints.csv", index=False)
dataset_fingerprint = hashlib.sha256("".join(fp["sha256"]).encode()).hexdigest()
log.info("Dataset fingerprint (SHA-256 over sorted file hashes): %s", dataset_fingerprint)

# Compare against provider checksums whenever we have them (S3 ETag = MD5 of single-part objects)
provider_verified = False
listings = [ART / f"provider_listing_{h}.csv" for h in CFG["hospital_dirs"]]
if all(p.exists() for p in listings):
    prov = pd.concat([pd.read_csv(p) for p in listings])
    prov["file"] = prov["key"].str.split("/").str[-1]
    prov = prov[~prov["etag"].astype(str).str.contains("-")]
    mrg = fp.merge(prov[["file", "etag", "size"]], on="file", how="left")
    n_cmp = mrg["etag"].notna().sum()
    n_bad = ((mrg["etag"].notna()) & (mrg["etag"] != mrg["md5"])).sum()
    provider_verified = check("provider_md5_etag_match", n_bad == 0 and n_cmp == len(fp),
                              f"compared={n_cmp}, mismatches={n_bad}")
else:
    check("provider_md5_etag_match", True,
          "SKIPPED: source not downloaded in this session (attached/local copy); structural checks only")

# Official SHA256SUMS.txt is tiny (covers only a few top-level files) - record it for completeness.
sha256sums_text = None
try:
    sha256sums_text = http_get(CFG["sha256sums_url"]).text
    (ART / "official_SHA256SUMS.txt").write_text(sha256sums_text)
except Exception as e:
    log.warning("Could not fetch official SHA256SUMS.txt (%s); not required.", e)

## 4. Parse all records and run structural checks
Each `.psv` is pipe-separated with a header. Checks: exact 41-column header, numeric parsing, hourly
`ICULOS` steps, binary and non-decreasing `SepsisLabel`, constant demographics within a patient.

In [ ]:
def read_psv(path: Path):
    with open(path) as f:
        header = f.readline().strip().split("|")
    df = pd.read_csv(path, sep="|", dtype=np.float32)
    return header, df

frames, problems = [], []
with StageTimer("parse", log):
    tasks = [(h, p) for h in ("A", "B") for p in files_by_hosp[h]]
    with ThreadPoolExecutor(8) as ex:
        results = list(ex.map(lambda hp: (hp, read_psv(hp[1])), tasks))
    for (h, p), (header, df) in results:
        if header != EXPECTED_COLUMNS:
            problems.append((p.name, "header_mismatch"))
            continue
        if len(df) == 0:
            problems.append((p.name, "empty"))
            continue
        df.insert(0, "t", np.arange(len(df), dtype=np.int16))
        df.insert(0, "hospital", h)
        df.insert(0, "patient_id", p.stem)
        frames.append(df)

check("all_headers_match_official_41_columns", not any(x[1] == "header_mismatch" for x in problems),
      [x for x in problems if x[1] == "header_mismatch"][:5])
check("no_empty_records", not any(x[1] == "empty" for x in problems))
hourly = pd.concat(frames, ignore_index=True)
del frames, results
hourly["patient_id"] = hourly["patient_id"].astype("category")
hourly["hospital"] = hourly["hospital"].astype("category")
log.info("Hourly table: %d rows x %d cols, %.0f MB", *hourly.shape, hourly.memory_usage(deep=True).sum() / 1e6)

In [ ]:
g = hourly.groupby("patient_id", observed=True, sort=False)

lab = hourly["SepsisLabel"]
check("sepsis_label_binary", set(np.unique(lab.dropna())) <= {0.0, 1.0} and lab.notna().all())
nondecr = g["SepsisLabel"].apply(lambda s: bool((np.diff(s.values) >= 0).all()))
check("sepsis_label_non_decreasing_per_patient", nondecr.all(), f"violations={(~nondecr).sum()}")

icu_step = g["ICULOS"].apply(lambda s: bool((np.diff(s.values) == 1).all()))
check("iculos_hourly_steps", icu_step.all(), f"patients with non-unit steps={(~icu_step).sum()} (reported, not fatal)")
checks[-1]["status"] = "PASS" if icu_step.all() else "WARN"

for col in ["Age", "Gender", "HospAdmTime"]:
    nun = g[col].nunique(dropna=True)
    ok = (nun <= 1).all()
    check(f"{col}_constant_within_patient", ok, f"patients with >1 value={(nun > 1).sum()}")
    if not ok:
        checks[-1]["status"] = "WARN"
check("gender_binary", set(np.unique(hourly["Gender"].dropna())) <= {0.0, 1.0})

## 5. Patient table (label timing follows the official scoring code)
The labels in the training data were already shifted six hours ahead: `SepsisLabel = 1` for `t ≥ t_sepsis − 6`.
As in `evaluate_sepsis_score.py`: `t_sepsis = argmax(SepsisLabel) − dt_optimal = first_positive_index + 6` (row index, hours).

In [ ]:
first_pos = g["SepsisLabel"].apply(lambda s: int(np.argmax(s.values)) if s.values.max() > 0 else -1)
patients = pd.DataFrame({
    "patient_id": first_pos.index.astype(str),
    "hospital": g["hospital"].first().astype(str).values,
    "n_rows": g.size().values,
    "septic": (first_pos.values >= 0).astype(np.int32),
    "first_label_idx": first_pos.values.astype(np.int32),
})
patients["t_sepsis_idx"] = np.where(patients["septic"] == 1,
                                    patients["first_label_idx"] - UTILITY_PARAMS["dt_optimal"], -1)
patients["iculos_first"] = g["ICULOS"].first().values
patients["iculos_last"] = g["ICULOS"].last().values
patients["age"] = g["Age"].first().values
patients["gender"] = g["Gender"].first().values
patients["hosp_adm_time"] = g["HospAdmTime"].first().values
patients = patients.sort_values(["hospital", "patient_id"]).reset_index(drop=True)

summary = patients.groupby("hospital").agg(patients=("patient_id", "size"), hours=("n_rows", "sum"),
                                           septic=("septic", "sum"))
summary = summary.astype(float)
summary["septic_pct"] = 100.0 * summary["septic"] / summary["patients"]
summary.loc["Total"] = [summary["patients"].sum(), summary["hours"].sum(), summary["septic"].sum(),
                        100 * summary["septic"].sum() / summary["patients"].sum()]
display(summary)
check("hourly_rows_match_patient_table", summary.loc["Total", "hours"] == len(hourly))
check("t_sepsis_consistent", ((patients["septic"] == 0) | (patients["t_sepsis_idx"] == patients["first_label_idx"] + 6)).all())

## 6. Save compact artifacts

In [ ]:
with StageTimer("save", log):
    p1 = save_table(hourly, ART / "physionet2019_hourly.parquet")
    p2 = save_table(patients, ART / "patients.parquet")
summary.reset_index().to_csv(RES / "tables" / "dataset_counts_by_hospital.csv", index=False)
log.info("Saved %s and %s", p1, p2)

## 7. Official scoring script (used later to cross-check our vectorised utility implementation)

In [ ]:
eval_status = "unavailable"
try:
    code = http_get(CFG["eval_script_url"]).text
    assert "def compute_prediction_utility" in code
    dst = ART / "evaluate_sepsis_score_official.py"
    dst.write_text(code)
    import importlib.util
    spec = importlib.util.spec_from_file_location("official_eval", dst)
    official_eval = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(official_eval)
    # sanity: a non-septic patient with no alarms has zero utility
    u0 = official_eval.compute_prediction_utility(np.zeros(20), np.zeros(20))
    assert abs(u0) < 1e-12
    eval_status = f"downloaded sha256={sha256_file(dst)}"
except Exception as e:
    log.warning("Official evaluation script not available (%s). Notebook 05 will still implement the "
                "utility exactly; cross-check will be skipped.", e)
check("official_eval_script", True, eval_status)

## 8. Dataset manifest, references and check report

In [ ]:
references_bib = r"""@article{reyna2020sepsis,
  author  = {Reyna, Matthew A. and Josef, Christopher S. and Jeter, Russell and Shashikumar, Supreeth P. and Westover, M. Brandon and Nemati, Shamim and Clifford, Gari D. and Sharma, Ashish},
  title   = {Early Prediction of Sepsis From Clinical Data: The {PhysioNet/Computing in Cardiology Challenge} 2019},
  journal = {Critical Care Medicine},
  volume  = {48},
  number  = {2},
  pages   = {210--217},
  year    = {2020},
  doi     = {10.1097/CCM.0000000000004145}
}
@article{goldberger2000physionet,
  author  = {Goldberger, Ary L. and Amaral, Luis A. N. and Glass, Leon and Hausdorff, Jeffrey M. and Ivanov, Plamen Ch. and Mark, Roger G. and Mietus, Joseph E. and Moody, George B. and Peng, Chung-Kang and Stanley, H. Eugene},
  title   = {{PhysioBank, PhysioToolkit, and PhysioNet}: Components of a New Research Resource for Complex Physiologic Signals},
  journal = {Circulation},
  volume  = {101},
  number  = {23},
  pages   = {e215--e220},
  year    = {2000},
  doi     = {10.1161/01.CIR.101.23.e215}
}
@article{singer2016sepsis3,
  author  = {Singer, Mervyn and Deutschman, Clifford S. and Seymour, Christopher Warren and others},
  title   = {The Third International Consensus Definitions for Sepsis and Septic Shock (Sepsis-3)},
  journal = {JAMA},
  volume  = {315},
  number  = {8},
  pages   = {801--810},
  year    = {2016},
  doi     = {10.1001/jama.2016.0287}
}
"""
(RES / "references.bib").write_text(references_bib)

manifest = {
    "dataset": CFG["dataset"], "landing_page": CFG["landing_page"], "license": CFG["license"],
    "required_citations": ["reyna2020sepsis", "goldberger2000physionet"],
    "source_mode": source_mode, "download_log": download_log,
    "provider_checksum_verified": provider_verified,
    "dataset_fingerprint_sha256": dataset_fingerprint,
    "official_sha256sums_txt_present": sha256sums_text is not None,
    "patients": {h: int(len(v)) for h, v in files_by_hosp.items()},
    "hourly_rows": int(len(hourly)),
    "septic_patients": {h: int(patients.loc[patients.hospital == h, "septic"].sum()) for h in ("A", "B")},
    "columns": EXPECTED_COLUMNS,
    "label_definition": ("SepsisLabel = 1 for t >= t_sepsis - 6 (labels shifted 6 h ahead by the challenge "
                         "organisers); t_sepsis defined by Sepsis-3; non-septic patients always 0."),
    "documented_unit_anomalies_to_verify_in_02": {
        "FiO2": "documented as %, verify fraction vs percent", "Lactate": "documented as mg/dL",
        "Magnesium": "documented as mmol/dL"},
    "official_eval_script": eval_status,
    "environment": env_info(),
}
save_json(manifest, RES / "dataset_manifest.json")
chk = pd.DataFrame(checks)
chk.to_csv(RES / "tables" / "integrity_checks.csv", index=False)
display(chk)

n_fail = (chk["status"] == "FAIL").sum()
if n_fail:
    raise RuntimeError(f"{n_fail} integrity checks FAILED - do not proceed. See results/tables/integrity_checks.csv")
print("\n✅ Dataset acquired and verified. Next: 02_data_profiling (attach this notebook's output as input).")

# 02 · Data profiling, unit verification and validation of physiological constraints
**Input:** output of `01_dataset_acquisition` (attach it via *Add Input → Your Work → Notebooks*).

**Purpose.** Characterise the cohort before any modelling decision, and test (rather than assume) the documented units
and the physiological relations that PTC-SSL will use as constraints.

1. Cohort description by hospital (Table 1 of the paper).
2. Missingness and measurement frequency per variable.
3. **Unit verification**: documented unit vs observed distribution. The official description lists FiO2 as %, Lactate as mg/dL and Magnesium as mmol/dL, which needs checking against the data.
4. Plausibility (QC) ranges in the verified units, with the counts they would remove. Saved for `03`.
5. **Constraint validation**: each candidate physiological relation is checked on same-hour co-measured raw values in **each hospital separately**, against acceptance criteria fixed *before* looking at the data (below). No outcome labels are used.
6. Evidence for choosing the window length.

**Pre-specified acceptance rule for a constraint:** at least `MIN_CO` co-measured rows in *both* hospitals, and
(equality relations) at least 70% of rows within tolerance, or (inequality relations) at most 5% violations, in *both* hospitals.

Runs on CPU in about 2–4 min.

In [ ]:
# ===== PTC-SSL common setup (identical in every notebook; do not edit per notebook) =====
import os, sys, json, time, math, glob, shutil, random, hashlib, logging, platform, warnings
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Kaggle paths (overridable through environment variables for local testing only)
WORK = Path(os.environ.get("PTCSSL_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("PTCSSL_INPUT", "/kaggle/input"))
ART = WORK / "artifacts"   # data artifacts consumed by later notebooks (may contain patient-level data)
RES = WORK / "results"     # metrics / tables / figures / configs (no raw patient data) -> paper package
LOGS = WORK / "logs"
for _d in (ART, RES, LOGS, RES / "figures", RES / "tables", RES / "configs"):
    _d.mkdir(parents=True, exist_ok=True)

GLOBAL_SEED = 2026

def set_seed(seed: int = GLOBAL_SEED):
    """Seed python, numpy and (if present) torch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed()

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
        fh = logging.FileHandler(LOGS / f"{name}.log", mode="a")
        fh.setFormatter(fmt)
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(fh)
        logger.addHandler(sh)
        logger.propagate = False
    return logger

try:
    display  # noqa: F821  (exists inside Jupyter/Kaggle)
except NameError:
    def display(x):
        print(x)

def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def md5_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    return path

def load_json(path):
    with open(path) as f:
        return json.load(f)

def save_table(df: pd.DataFrame, path):
    """Save as parquet (Kaggle has pyarrow); fall back to pickle if parquet is unavailable."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except ImportError:
        alt = path.with_suffix(".pkl")
        df.to_pickle(alt)
        return alt

def load_table(path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix == ".parquet" and path.exists():
        return pd.read_parquet(path)
    alt = path.with_suffix(".pkl")
    if alt.exists():
        return pd.read_pickle(alt)
    if path.exists():
        return pd.read_pickle(path)
    raise FileNotFoundError(path)

_ARTIFACT_CACHE = {}

def find_artifact(name: str, producer: str) -> Path:
    """Locate an artifact produced by an earlier notebook.
    Search order: this session's /kaggle/working/artifacts, then any attached input (/kaggle/input/**).
    Table artifacts may exist as .parquet or .pkl."""
    if name in _ARTIFACT_CACHE:
        return _ARTIFACT_CACHE[name]
    stem, suffix = os.path.splitext(name)
    names = [name] + ([stem + ".pkl"] if suffix == ".parquet" else [])
    for n in names:
        p = ART / n
        if p.exists():
            _ARTIFACT_CACHE[name] = p
            return p
    hits = []
    if INPUT.exists():
        for dirpath, _, files in os.walk(INPUT, followlinks=True):
            if Path(dirpath).name == "artifacts":          # never pick up copies under results/
                hits += [Path(dirpath) / n for n in names if n in files]
    if hits:
        hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        if len(hits) > 1:
            print(f"[find_artifact] {len(hits)} copies of {name} found; using newest: {hits[0]}")
        _ARTIFACT_CACHE[name] = hits[0]
        return hits[0]
    raise FileNotFoundError(
        f"Artifact '{name}' not found. Run notebook {producer} first, then attach its output to this "
        f"notebook (Kaggle: Add Input -> Your Work -> Notebooks -> {producer})."
    )

def env_info() -> dict:
    info = {
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
    for mod in ("scipy", "sklearn", "matplotlib", "torch", "lightgbm"):
        try:
            m = __import__(mod)
            info[mod] = getattr(m, "__version__", "unknown")
        except ImportError:
            info[mod] = "not installed"
    try:
        import torch
        info["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return info

class StageTimer:
    """Context manager that logs wall-clock time of a stage (also used for the compute-cost table)."""
    def __init__(self, name, logger=None):
        self.name, self.logger = name, logger
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        self.seconds = time.time() - self.t0
        msg = f"[time] {self.name}: {self.seconds:.1f}s"
        (self.logger.info if self.logger else print)(msg)

# PhysioNet 2019 schema (verified against the official data description, physionet.org/content/challenge-2019/1.0.0/)
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos", "Calcium",
        "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate", "Magnesium", "Phosphate",
        "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets"]
DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
DYNAMIC_VARS = VITALS + LABS                       # 34 time-varying clinical variables
EXPECTED_COLUMNS = DYNAMIC_VARS + DEMOGRAPHICS + ["SepsisLabel"]   # 41 columns, official order

# Official utility parameters (physionetchallenges/evaluation-2019, evaluate_sepsis_score.py)
UTILITY_PARAMS = dict(dt_early=-12, dt_optimal=-6, dt_late=3, max_u_tp=1.0, min_u_fn=-2.0, u_fp=-0.05, u_tn=0.0)
print(f"[setup] WORK={WORK}  INPUT={INPUT}  seed={GLOBAL_SEED}")

In [ ]:
NB = "02_data_profiling"
log = get_logger(NB)
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False})

hourly = load_table(find_artifact("physionet2019_hourly.parquet", "01_dataset_acquisition"))
patients = load_table(find_artifact("patients.parquet", "01_dataset_acquisition"))
assert list(hourly.columns[3:]) == EXPECTED_COLUMNS, "unexpected schema - rerun 01"
assert hourly["patient_id"].nunique() == len(patients)
MIN_CO = int(os.environ.get("PTCSSL_MIN_CO", 500))   # minimum co-measured rows per hospital
HOSPS = ["A", "B"]
log.info("Loaded %d hourly rows, %d patients", len(hourly), len(patients))

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(RES / "figures" / f"{name}.{ext}", bbox_inches="tight")
    plt.show() if matplotlib.get_backend().lower() != "agg" else None
    plt.close(fig)

## 1. Cohort description (Table 1)

In [ ]:
def med_iqr(x):
    x = pd.Series(x).dropna()
    return f"{x.median():.1f} [{x.quantile(.25):.1f}–{x.quantile(.75):.1f}]" if len(x) else "NA"

unit_first = hourly.groupby("patient_id", observed=True)[["Unit1", "Unit2"]].first()
patients = patients.merge(unit_first, left_on="patient_id", right_index=True, how="left")
rows = []
for h in HOSPS + ["All"]:
    p = patients if h == "All" else patients[patients.hospital == h]
    s = p[p.septic == 1]
    rows.append({
        "cohort": h, "patients": len(p), "patient_hours": int(p.n_rows.sum()),
        "ICU hours per record, median [IQR]": med_iqr(p.n_rows),
        "age, median [IQR]": med_iqr(p.age), "male, %": f"{100 * p.gender.mean():.1f}",
        "MICU / SICU / unit missing, %": f"{100*(p.Unit1==1).mean():.1f} / {100*(p.Unit2==1).mean():.1f} / {100*p.Unit1.isna().mean():.1f}",
        "septic, n (%)": f"{len(s)} ({100 * len(s) / len(p):.2f})",
        "t_sepsis (row h), median [IQR]": med_iqr(s.t_sepsis_idx),
        "septic with label at first row, n": int((s.first_label_idx == 0).sum()),
        "septic with first label <= 6 h, n": int((s.first_label_idx <= 6).sum()),
    })
table1 = pd.DataFrame(rows).set_index("cohort").T
table1.to_csv(RES / "tables" / "table1_cohort.csv")
display(table1)

## 2. Missingness and measurement frequency

In [ ]:
obs = hourly[DYNAMIC_VARS].notna()
miss_rows = []
for h in HOSPS:
    sel = (hourly.hospital == h).values
    o = obs[sel]
    pid = hourly.loc[sel, "patient_id"]
    ever = o.groupby(pid.values).any().mean()
    for v in DYNAMIC_VARS:
        # median gap between consecutive observations (hours), pooled over patients
        tv = hourly.loc[sel & obs[v].values, ["patient_id", "t"]]
        gaps = tv.groupby("patient_id", observed=True)["t"].diff().dropna()
        miss_rows.append({"hospital": h, "variable": v, "group": "vital" if v in VITALS else "lab",
                          "pct_hours_observed": 100 * o[v].mean(), "pct_patients_ever_observed": 100 * ever[v],
                          "median_gap_h": float(gaps.median()) if len(gaps) else np.nan})
miss = pd.DataFrame(miss_rows)
miss.to_csv(RES / "tables" / "missingness_by_variable.csv", index=False)
display(miss.pivot(index="variable", columns="hospital", values="pct_hours_observed").round(2).loc[DYNAMIC_VARS])

fig, ax = plt.subplots(figsize=(7.2, 3.2))
pv = miss.pivot(index="variable", columns="hospital", values="pct_hours_observed").loc[DYNAMIC_VARS]
x = np.arange(len(pv))
ax.bar(x - 0.2, pv["A"], 0.4, label="Hospital A")
ax.bar(x + 0.2, pv["B"], 0.4, label="Hospital B")
ax.set_xticks(x, pv.index, rotation=90)
ax.set_ylabel("Hours with a measurement (%)")
ax.set_yscale("log")
ax.legend(frameon=False)
savefig(fig, "fig_missingness_by_variable")

## 3. Unit verification
For each variable the observed median is compared with the reference median window expected under each candidate unit.
The windows are broad adult ICU ranges, chosen only to tell units apart (a factor of 10 or more between candidates), not to define normality.

In [ ]:
DOC_UNITS = {"HR": "bpm", "O2Sat": "%", "Temp": "°C", "SBP": "mmHg", "MAP": "mmHg", "DBP": "mmHg",
             "Resp": "breaths/min", "EtCO2": "mmHg", "BaseExcess": "mmol/L", "HCO3": "mmol/L", "FiO2": "%",
             "pH": "-", "PaCO2": "mmHg", "SaO2": "%", "AST": "IU/L", "BUN": "mg/dL", "Alkalinephos": "IU/L",
             "Calcium": "mg/dL", "Chloride": "mmol/L", "Creatinine": "mg/dL", "Bilirubin_direct": "mg/dL",
             "Glucose": "mg/dL", "Lactate": "mg/dL", "Magnesium": "mmol/dL", "Phosphate": "mg/dL",
             "Potassium": "mmol/L", "Bilirubin_total": "mg/dL", "TroponinI": "ng/mL", "Hct": "%", "Hgb": "g/dL",
             "PTT": "s", "WBC": "10^3/µL", "Fibrinogen": "mg/dL", "Platelets": "10^3/µL"}
# candidate units -> (low, high) window for the cohort MEDIAN
CANDIDATES = {
    "FiO2": {"fraction": (0.21, 1.0), "%": (21, 100)},
    "Lactate": {"mmol/L": (0.5, 6), "mg/dL": (4.5, 54)},
    "Magnesium": {"mg/dL": (1.4, 3.0), "mmol/L": (0.55, 1.25), "mmol/dL": (0.055, 0.125)},
    "Calcium": {"mg/dL (total)": (6, 12), "mmol/L (ionised)": (0.8, 1.6)},
    "Temp": {"°C": (34, 40), "°F": (93, 104)},
    "Hct": {"%": (18, 55), "fraction": (0.18, 0.55)},
    "Hgb": {"g/dL": (6, 18), "g/L": (60, 180), "mmol/L": (3.7, 11.2)},
    "Glucose": {"mg/dL": (60, 300), "mmol/L": (3.3, 16.7)},
    "Creatinine": {"mg/dL": (0.3, 6), "µmol/L": (27, 530)},
    "BUN": {"mg/dL": (5, 100), "mmol/L urea": (1.8, 36)},
    "Bilirubin_total": {"mg/dL": (0.1, 10), "µmol/L": (1.7, 171)},
    "PaCO2": {"mmHg": (25, 70), "kPa": (3.3, 9.3)},
    "SaO2": {"%": (80, 100), "fraction": (0.8, 1.0)},
    "O2Sat": {"%": (80, 100), "fraction": (0.8, 1.0)},
}
unit_rows = []
for v in DYNAMIC_VARS:
    for h in HOSPS:
        x = hourly.loc[hourly.hospital == h, v].dropna()
        r = {"variable": v, "hospital": h, "documented_unit": DOC_UNITS[v], "n": len(x),
             "p01": x.quantile(.01) if len(x) else np.nan, "median": x.median() if len(x) else np.nan,
             "p99": x.quantile(.99) if len(x) else np.nan}
        if v in CANDIDATES and len(x):
            fits = [u for u, (lo, hi) in CANDIDATES[v].items() if lo <= r["median"] <= hi]
            doc_fits = [u for u in fits if u.split(" ")[0] == DOC_UNITS[v]]
            if len(fits) == 1:
                r["inferred_unit"] = fits[0]
            elif len(fits) > 1 and len(doc_fits) == 1:   # overlapping windows: data consistent with documentation
                r["inferred_unit"] = doc_fits[0]
            else:
                r["inferred_unit"] = ("ambiguous: " + ",".join(fits)) if fits else "none"
        else:
            r["inferred_unit"] = DOC_UNITS[v] + " (documented; not tested)"
        if v == "FiO2" and len(x):
            r["pct_values_gt_1"] = 100 * (x > 1).mean()
        unit_rows.append(r)
units = pd.DataFrame(unit_rows)
units.to_csv(RES / "tables" / "unit_verification.csv", index=False)
display(units[units.variable.isin(CANDIDATES)].round(3))

def inferred(v):
    u = units[(units.variable == v)]["inferred_unit"].unique()
    if len(u) != 1:
        log.warning("Unit for %s differs between hospitals or is ambiguous: %s", v, u)
    return u[0]

VERIFIED_UNITS = {v: inferred(v) for v in CANDIDATES}
for v, u in VERIFIED_UNITS.items():
    flag = "" if u.split(" ")[0] in DOC_UNITS[v] or DOC_UNITS[v] in u else "   <-- differs from documentation"
    print(f"{v:16s} documented={DOC_UNITS[v]:9s} inferred={u}{flag}")

## 4. Plausibility (QC) ranges in the verified units
Values outside these ranges are physiologically implausible (device or entry errors) and are set to missing in `03`.
FiO2 recorded as a percentage (1 < value ≤ 100) is converted to a fraction **only if** the dominant verified unit is a fraction.

In [ ]:
QC = {"HR": (20, 300), "O2Sat": (50, 100), "Temp": (25, 45), "SBP": (40, 300), "MAP": (20, 250), "DBP": (10, 200),
      "Resp": (3, 80), "EtCO2": (0, 100), "BaseExcess": (-40, 40), "HCO3": (2, 60), "FiO2": (0.21, 1.0),
      "pH": (6.5, 8.0), "PaCO2": (5, 150), "SaO2": (30, 100), "AST": (1, 20000), "BUN": (1, 300),
      "Alkalinephos": (1, 5000), "Calcium": (2, 20), "Chloride": (50, 160), "Creatinine": (0.1, 30),
      "Bilirubin_direct": (0, 50), "Glucose": (10, 2000), "Lactate": (0.1, 40), "Magnesium": (0.3, 10),
      "Phosphate": (0.2, 20), "Potassium": (1, 12), "Bilirubin_total": (0, 80), "TroponinI": (0, 500),
      "Hct": (5, 75), "Hgb": (1, 25), "PTT": (10, 250), "WBC": (0, 500), "Fibrinogen": (20, 2000),
      "Platelets": (1, 2000)}
# ranges above assume: FiO2 fraction, Lactate mmol/L, Magnesium mg/dL, Calcium mg/dL, Temp °C.
ASSUMED = {"FiO2": "fraction", "Lactate": "mmol/L", "Magnesium": "mg/dL", "Calcium": "mg/dL (total)", "Temp": "°C",
           "Hct": "%", "Hgb": "g/dL", "Glucose": "mg/dL", "Creatinine": "mg/dL", "BUN": "mg/dL",
           "Bilirubin_total": "mg/dL", "PaCO2": "mmHg", "SaO2": "%", "O2Sat": "%"}
mismatch = {v: (VERIFIED_UNITS[v], ASSUMED[v]) for v in ASSUMED if VERIFIED_UNITS[v] != ASSUMED[v]}
if mismatch:
    log.error("Verified units differ from the units assumed by the QC ranges: %s", mismatch)
    raise RuntimeError(f"Unit mismatch {mismatch}: update QC/ASSUMED before continuing (do not ignore).")
FIO2_PERCENT_FIX = VERIFIED_UNITS["FiO2"] == "fraction"

def apply_fio2_fix(s):
    s = s.copy()
    m = (s > 1) & (s <= 100)
    s[m] = s[m] / 100.0
    return s, int(m.sum())

qc_rows = []
for h in HOSPS:
    sel = hourly.hospital == h
    for v in DYNAMIC_VARS:
        x = hourly.loc[sel, v]
        n_fix = 0
        if v == "FiO2" and FIO2_PERCENT_FIX:
            x, n_fix = apply_fio2_fix(x)
        lo, hi = QC[v]
        n_obs = int(x.notna().sum())
        n_out = int(((x < lo) | (x > hi)).sum())
        qc_rows.append({"hospital": h, "variable": v, "unit": ASSUMED.get(v, DOC_UNITS[v]), "low": lo, "high": hi,
                        "n_observed": n_obs, "n_out_of_range": n_out,
                        "pct_out_of_range": 100 * n_out / max(n_obs, 1), "n_fio2_percent_converted": n_fix})
qc = pd.DataFrame(qc_rows)
qc.to_csv(RES / "tables" / "qc_out_of_range.csv", index=False)
display(qc[qc.n_out_of_range > 0].sort_values("pct_out_of_range", ascending=False).head(20))
save_json({"ranges": QC, "units": {v: ASSUMED.get(v, DOC_UNITS[v]) for v in DYNAMIC_VARS},
           "fio2_percent_to_fraction": FIO2_PERCENT_FIX}, ART / "qc_ranges.json")
save_json({"ranges": QC, "verified_units": VERIFIED_UNITS}, RES / "configs" / "qc_ranges.json")

## 5. Validation of candidate physiological constraints
Checked on **raw, same-hour, co-measured** values after QC. No forward filling, no labels.

| id | relation | units | tolerance | type |
|---|---|---|---|---|
| C1 | MAP ≈ (SBP + 2·DBP)/3 | mmHg | 10 mmHg | equality |
| C2 | SBP ≥ MAP ≥ DBP | mmHg | — | inequality |
| C3 | pH ≈ 6.1 + log10(HCO3 / (0.03·PaCO2)) (Henderson–Hasselbalch) | mmol/L, mmHg | 0.05 | equality |
| C4 | Hct ≈ 3·Hgb | %, g/dL | 3 % points | equality |
| C5 | Bilirubin_direct ≤ Bilirubin_total | mg/dL | — | inequality |
| C6 | SaO2 ≈ O2Sat (arterial vs pulse oximetry) | % | 4 % points | equality |

The HCO3 column may be a chemistry-panel bicarbonate (total CO2) rather than the blood-gas calculated value.
C3 therefore tests whether the relation holds *in this dataset*, which is the point of validating it.

In [ ]:
qc_ranges = load_json(ART / "qc_ranges.json")
def qc_clean(df):
    out = df.copy()
    for v in DYNAMIC_VARS:
        if v not in out:
            continue
        if v == "FiO2" and qc_ranges["fio2_percent_to_fraction"]:
            out[v], _ = apply_fio2_fix(out[v])
        lo, hi = qc_ranges["ranges"][v]
        out.loc[(out[v] < lo) | (out[v] > hi), v] = np.nan
    return out

CONSTRAINTS = {
    "C1_MAP_formula": dict(vars=["MAP", "SBP", "DBP"], kind="eq", tol=10.0, unit="mmHg",
                           f=lambda d: d.MAP - (d.SBP + 2 * d.DBP) / 3),
    "C2_BP_ordering": dict(vars=["SBP", "MAP", "DBP"], kind="ineq", tol=0.0, unit="mmHg",
                           f=lambda d: np.maximum(d.MAP - d.SBP, d.DBP - d.MAP)),   # >0 = violation
    "C3_Henderson_Hasselbalch": dict(vars=["pH", "HCO3", "PaCO2"], kind="eq", tol=0.05, unit="pH units",
                                     f=lambda d: d.pH - (6.1 + np.log10(d.HCO3 / (0.03 * d.PaCO2)))),
    "C4_Hct_3xHgb": dict(vars=["Hct", "Hgb"], kind="eq", tol=3.0, unit="% points",
                         f=lambda d: d.Hct - 3 * d.Hgb),
    "C5_bilirubin_direct_le_total": dict(vars=["Bilirubin_direct", "Bilirubin_total"], kind="ineq", tol=0.0,
                                         unit="mg/dL", f=lambda d: d.Bilirubin_direct - d.Bilirubin_total),
    "C6_SaO2_vs_SpO2": dict(vars=["SaO2", "O2Sat"], kind="eq", tol=4.0, unit="% points",
                            f=lambda d: d.SaO2 - d.O2Sat),
}
clean = qc_clean(hourly[["hospital"] + DYNAMIC_VARS])
cv_rows, residuals = [], {}
for cid, c in CONSTRAINTS.items():
    for h in HOSPS:
        d = clean.loc[clean.hospital == h, c["vars"]].dropna()
        r = c["f"](d) if len(d) else pd.Series(dtype=float)
        residuals[(cid, h)] = r.values
        row = {"constraint": cid, "hospital": h, "n_co_measured": len(d), "unit": c["unit"], "type": c["kind"],
               "tolerance": c["tol"]}
        if len(d):
            if c["kind"] == "eq":
                row.update(median_residual=float(r.median()), mad=float((r - r.median()).abs().median()),
                           pct_within_tol=float(100 * (r.abs() <= c["tol"]).mean()))
                row["pass"] = bool(len(d) >= MIN_CO and row["pct_within_tol"] >= 70)
            else:
                row.update(pct_violations=float(100 * (r > c["tol"]).mean()))
                row["pass"] = bool(len(d) >= MIN_CO and row["pct_violations"] <= 5)
        else:
            row["pass"] = False
        cv_rows.append(row)
cv = pd.DataFrame(cv_rows)
cv.to_csv(RES / "tables" / "constraint_validation.csv", index=False)
display(cv.round(3))

accepted = [cid for cid in CONSTRAINTS if cv[(cv.constraint == cid)]["pass"].all()]
rejected = [cid for cid in CONSTRAINTS if cid not in accepted]
log.info("Accepted constraints (both hospitals): %s", accepted)
log.info("Rejected constraints: %s", rejected)
save_json({"accepted": accepted, "rejected": rejected, "min_co_measured": MIN_CO,
           "definitions": {cid: {k: v for k, v in c.items() if k != "f"} for cid, c in CONSTRAINTS.items()},
           "validation": cv.to_dict(orient="records")},
          ART / "constraints_validated.json")
save_json({"accepted": accepted, "rejected": rejected}, RES / "configs" / "constraints_validated.json")
if not accepted:
    log.warning("No constraint passed. PTC-SSL physiology loss would be empty: revisit before notebook 07.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.4))
for ax, (cid, c) in zip(axes.ravel(), CONSTRAINTS.items()):
    for h in HOSPS:
        r = residuals[(cid, h)]
        if len(r):
            lo, hi = np.nanpercentile(r, [1, 99])
            ax.hist(np.clip(r, lo, hi), bins=60, alpha=0.55, density=True, label=h)
    if c["kind"] == "eq":
        for s in (-1, 1):
            ax.axvline(s * c["tol"], color="k", lw=0.8, ls="--")
    else:
        ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_title(cid.replace("_", " "), fontsize=8)
    ax.set_xlabel(f"residual ({c['unit']})", fontsize=7)
axes[0, 0].legend(frameon=False, fontsize=7)
fig.tight_layout()
savefig(fig, "fig_constraint_residuals")

## 6. Evidence for the temporal window length
For a window of W hours, how many prediction hours have a full W-hour history? This informs (but does not tune) the choice of W in `04`.

In [ ]:
t = hourly["t"].values
wrows = []
for W in (6, 12, 24, 48, 72):
    for h in HOSPS:
        sel = (hourly.hospital == h).values
        wrows.append({"W_hours": W, "hospital": h, "pct_prediction_hours_with_full_history": 100 * (t[sel] >= W - 1).mean()})
wl = pd.DataFrame(wrows)
wl.to_csv(RES / "tables" / "window_length_coverage.csv", index=False)
display(wl.pivot(index="W_hours", columns="hospital").round(1))
display(patients.groupby("hospital").n_rows.describe(percentiles=[.1, .25, .5, .75, .9]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6))
for h in HOSPS:
    axes[0].hist(patients.loc[patients.hospital == h, "n_rows"].clip(upper=200), bins=50, alpha=.55, label=h)
    axes[1].hist(patients.loc[(patients.hospital == h) & (patients.septic == 1), "t_sepsis_idx"].clip(upper=200),
                 bins=40, alpha=.55, label=h)
axes[0].set_xlabel("Hours of ICU data per record (clipped at 200)")
axes[1].set_xlabel("Sepsis onset t_sepsis (h from record start)")
axes[0].legend(frameon=False)
fig.tight_layout()
savefig(fig, "fig_los_and_onset")

## 7. Profiling report

In [ ]:
def md_table(df):
    try:
        return df.to_markdown()
    except ImportError:
        return "```\n" + df.to_string() + "\n```"

report = f"""# Data profiling report (PhysioNet/CinC 2019)
Generated: {dt.datetime.now(dt.timezone.utc).isoformat()}

## Cohort
{md_table(table1)}

## Unit verification (variables with ambiguous documentation)
{chr(10).join(f'- {v}: documented {DOC_UNITS[v]} -> verified {u}' for v, u in VERIFIED_UNITS.items())}

FiO2 percent-to-fraction conversion applied: {FIO2_PERCENT_FIX}

## Physiological constraints (pre-specified acceptance rule, both hospitals)
Accepted: {accepted}
Rejected: {rejected}

## Files
tables/table1_cohort.csv, tables/missingness_by_variable.csv, tables/unit_verification.csv,
tables/qc_out_of_range.csv, tables/constraint_validation.csv, tables/window_length_coverage.csv,
figures/fig_missingness_by_variable.*, figures/fig_constraint_residuals.*, figures/fig_los_and_onset.*
"""
(RES / "data_profile_report.md").write_text(report)
print(report)
print("✅ Profiling complete. Next: 03_preprocessing (attach outputs of 01 and 02).")

# 03 · Preprocessing: cohort, QC, patient-level splits, train-only normalisation
**Inputs:** outputs of `01_dataset_acquisition` and `02_data_profiling`.

**Design decisions (fixed before modelling):**
- **Cohort exclusion E1:** septic patients whose `SepsisLabel` is already 1 at the first ICU row (onset ≤ 6 h after the record starts), because early prediction is not defined for them. Counts are reported, and the official-utility sensitivity analysis in `11` can include them again.
- **QC:** implausible values are set to missing using the unit-verified ranges from `02`. FiO2 percentages are converted to fractions only if `02` verified this.
- **Features:** 34 time-varying variables, plus Age, Gender and log(ICULOS). `Unit1/Unit2/HospAdmTime` are left out: they are administrative and hospital-specific, so they could act as shortcuts under the A→B / B→A shift.
- **Splits (patient-level, stratified by hospital × septic, fixed seed):**
  - `pooled`: A∪B → 70 / 15 / 15 train / val / test.
  - `A2B`: A → 85 / 15 train / val; test = all eligible B.
  - `B2A`: B → 85 / 15 train / val; test = all eligible A.
- **SSL pretraining pool = the training patients of each protocol only.** Validation and test patients never enter pretraining.
- **Normalisation:** fitted per protocol on *training patients only*, using raw observations (each measurement counted once, no forward-filled copies). Skewed labs are log1p-transformed, then winsorised at the training 0.5/99.5 percentiles and z-scored.
- **Stored representation:** raw QC'd values (NaN = missing), observation mask, causal last-observation-carried-forward (LOCF) values, and hours since the last observation. LOCF and age are strictly causal (no back-filling).

Runs on CPU in about 3–6 min.

In [ ]:
# ===== PTC-SSL common setup (identical in every notebook; do not edit per notebook) =====
import os, sys, json, time, math, glob, shutil, random, hashlib, logging, platform, warnings
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Kaggle paths (overridable through environment variables for local testing only)
WORK = Path(os.environ.get("PTCSSL_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("PTCSSL_INPUT", "/kaggle/input"))
ART = WORK / "artifacts"   # data artifacts consumed by later notebooks (may contain patient-level data)
RES = WORK / "results"     # metrics / tables / figures / configs (no raw patient data) -> paper package
LOGS = WORK / "logs"
for _d in (ART, RES, LOGS, RES / "figures", RES / "tables", RES / "configs"):
    _d.mkdir(parents=True, exist_ok=True)

GLOBAL_SEED = 2026

def set_seed(seed: int = GLOBAL_SEED):
    """Seed python, numpy and (if present) torch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed()

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
        fh = logging.FileHandler(LOGS / f"{name}.log", mode="a")
        fh.setFormatter(fmt)
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(fh)
        logger.addHandler(sh)
        logger.propagate = False
    return logger

try:
    display  # noqa: F821  (exists inside Jupyter/Kaggle)
except NameError:
    def display(x):
        print(x)

def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def md5_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    return path

def load_json(path):
    with open(path) as f:
        return json.load(f)

def save_table(df: pd.DataFrame, path):
    """Save as parquet (Kaggle has pyarrow); fall back to pickle if parquet is unavailable."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except ImportError:
        alt = path.with_suffix(".pkl")
        df.to_pickle(alt)
        return alt

def load_table(path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix == ".parquet" and path.exists():
        return pd.read_parquet(path)
    alt = path.with_suffix(".pkl")
    if alt.exists():
        return pd.read_pickle(alt)
    if path.exists():
        return pd.read_pickle(path)
    raise FileNotFoundError(path)

_ARTIFACT_CACHE = {}

def find_artifact(name: str, producer: str) -> Path:
    """Locate an artifact produced by an earlier notebook.
    Search order: this session's /kaggle/working/artifacts, then any attached input (/kaggle/input/**).
    Table artifacts may exist as .parquet or .pkl."""
    if name in _ARTIFACT_CACHE:
        return _ARTIFACT_CACHE[name]
    stem, suffix = os.path.splitext(name)
    names = [name] + ([stem + ".pkl"] if suffix == ".parquet" else [])
    for n in names:
        p = ART / n
        if p.exists():
            _ARTIFACT_CACHE[name] = p
            return p
    hits = []
    if INPUT.exists():
        for dirpath, _, files in os.walk(INPUT, followlinks=True):
            if Path(dirpath).name == "artifacts":          # never pick up copies under results/
                hits += [Path(dirpath) / n for n in names if n in files]
    if hits:
        hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        if len(hits) > 1:
            print(f"[find_artifact] {len(hits)} copies of {name} found; using newest: {hits[0]}")
        _ARTIFACT_CACHE[name] = hits[0]
        return hits[0]
    raise FileNotFoundError(
        f"Artifact '{name}' not found. Run notebook {producer} first, then attach its output to this "
        f"notebook (Kaggle: Add Input -> Your Work -> Notebooks -> {producer})."
    )

def env_info() -> dict:
    info = {
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
    for mod in ("scipy", "sklearn", "matplotlib", "torch", "lightgbm"):
        try:
            m = __import__(mod)
            info[mod] = getattr(m, "__version__", "unknown")
        except ImportError:
            info[mod] = "not installed"
    try:
        import torch
        info["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return info

class StageTimer:
    """Context manager that logs wall-clock time of a stage (also used for the compute-cost table)."""
    def __init__(self, name, logger=None):
        self.name, self.logger = name, logger
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        self.seconds = time.time() - self.t0
        msg = f"[time] {self.name}: {self.seconds:.1f}s"
        (self.logger.info if self.logger else print)(msg)

# PhysioNet 2019 schema (verified against the official data description, physionet.org/content/challenge-2019/1.0.0/)
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos", "Calcium",
        "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate", "Magnesium", "Phosphate",
        "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets"]
DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
DYNAMIC_VARS = VITALS + LABS                       # 34 time-varying clinical variables
EXPECTED_COLUMNS = DYNAMIC_VARS + DEMOGRAPHICS + ["SepsisLabel"]   # 41 columns, official order

# Official utility parameters (physionetchallenges/evaluation-2019, evaluate_sepsis_score.py)
UTILITY_PARAMS = dict(dt_early=-12, dt_optimal=-6, dt_late=3, max_u_tp=1.0, min_u_fn=-2.0, u_fp=-0.05, u_tn=0.0)
print(f"[setup] WORK={WORK}  INPUT={INPUT}  seed={GLOBAL_SEED}")

In [ ]:
NB = "03_preprocessing"
log = get_logger(NB)
hourly = load_table(find_artifact("physionet2019_hourly.parquet", "01_dataset_acquisition"))
patients = load_table(find_artifact("patients.parquet", "01_dataset_acquisition"))
qc_cfg = load_json(find_artifact("qc_ranges.json", "02_data_profiling"))
cons = load_json(find_artifact("constraints_validated.json", "02_data_profiling"))

CFG = {
    "exclude_label_at_first_row": True,
    "split_seed": 2026,
    "pooled_fracs": {"train": 0.70, "val": 0.15, "test": 0.15},
    "external_train_frac": 0.85,
    "static_features": ["Age", "Gender", "log_ICULOS"],
    "excluded_columns": ["Unit1", "Unit2", "HospAdmTime"],
    "log1p_vars": ["AST", "BUN", "Alkalinephos", "Creatinine", "Bilirubin_direct", "Bilirubin_total", "Glucose",
                   "Lactate", "TroponinI", "WBC", "Fibrinogen", "Platelets", "PTT"],
    "winsor_pct": [0.5, 99.5],
    "max_age_h": 168.0,
}
ORGAN_SYSTEMS = {   # every dynamic variable belongs to exactly one system
    "cardiovascular": ["HR", "SBP", "MAP", "DBP", "TroponinI"],
    "respiratory": ["O2Sat", "Resp", "EtCO2", "FiO2", "SaO2", "PaCO2"],
    "acid_base_perfusion": ["pH", "HCO3", "BaseExcess", "Lactate"],
    "renal_electrolyte": ["BUN", "Creatinine", "Potassium", "Chloride"],
    "hepatic": ["AST", "Alkalinephos", "Bilirubin_direct", "Bilirubin_total"],
    "hematologic_coagulation": ["Hct", "Hgb", "Platelets", "PTT", "Fibrinogen"],
    "metabolic": ["Glucose", "Calcium", "Magnesium", "Phosphate"],
    "inflammation_thermoregulation": ["Temp", "WBC"],
}
flat = [v for vs in ORGAN_SYSTEMS.values() for v in vs]
assert sorted(flat) == sorted(DYNAMIC_VARS) and len(flat) == len(set(flat)) == 34, "organ map must partition the 34 vars"
print(json.dumps(CFG, indent=1))

## 1. Cohort and exclusions

In [ ]:
patients = patients.sort_values(["hospital", "patient_id"]).reset_index(drop=True)
patients["excl_label_at_first_row"] = (patients.septic == 1) & (patients.first_label_idx == 0)
patients["eligible"] = ~patients["excl_label_at_first_row"] if CFG["exclude_label_at_first_row"] else True
flow = patients.groupby("hospital").agg(records=("patient_id", "size"),
                                        excluded_E1=("excl_label_at_first_row", "sum"),
                                        eligible=("eligible", "sum"))
flow["eligible_septic"] = patients[patients.eligible].groupby("hospital").septic.sum()
display(flow)
flow.to_csv(RES / "tables" / "cohort_flow.csv")

## 2. Patient-level stratified splits

In [ ]:
def strat_split(df, fracs, seed):
    """Assign split names to rows of df, stratified by (hospital, septic). Exact, deterministic."""
    rng = np.random.default_rng(seed)
    out = pd.Series(index=df.index, dtype=object)
    names = list(fracs)
    for _, grp in df.groupby(["hospital", "septic"]):
        idx = grp.index.values.copy()
        rng.shuffle(idx)
        n = len(idx)
        cuts = np.floor(np.cumsum([fracs[k] for k in names]) * n + 1e-9).astype(int)
        cuts[-1] = n
        start = 0
        for name, c in zip(names, cuts):
            out.loc[idx[start:c]] = name
            start = c
    return out

elig = patients[patients.eligible]
for proto in ("pooled", "A2B", "B2A"):
    patients[f"split_{proto}"] = "excluded"
patients.loc[elig.index, "split_pooled"] = strat_split(elig, CFG["pooled_fracs"], CFG["split_seed"])
ext = {"train": CFG["external_train_frac"], "val": 1 - CFG["external_train_frac"]}
for proto, src, tgt in (("A2B", "A", "B"), ("B2A", "B", "A")):
    s = elig[elig.hospital == src]
    patients.loc[s.index, f"split_{proto}"] = strat_split(s, ext, CFG["split_seed"])
    patients.loc[elig[elig.hospital == tgt].index, f"split_{proto}"] = "test"

PROTOCOLS = ["pooled", "A2B", "B2A"]
split_tab = []
for proto in PROTOCOLS:
    col = f"split_{proto}"
    for sp in ("train", "val", "test"):
        p = patients[patients[col] == sp]
        split_tab.append({"protocol": proto, "split": sp, "patients": len(p), "septic": int(p.septic.sum()),
                          "septic_pct": 100 * p.septic.mean() if len(p) else np.nan,
                          "hospitals": ",".join(sorted(p.hospital.unique()))})
    # leakage guards
    sets = {sp: set(patients.loc[patients[col] == sp, "patient_id"]) for sp in ("train", "val", "test")}
    assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"])
    assert patients.loc[~patients.eligible, col].eq("excluded").all()
assert set(patients.loc[patients.split_A2B == "test", "hospital"]) <= {"B"}
assert set(patients.loc[patients.split_A2B.isin(["train", "val"]), "hospital"]) <= {"A"}
assert set(patients.loc[patients.split_B2A == "test", "hospital"]) <= {"A"}
split_tab = pd.DataFrame(split_tab)
split_tab.to_csv(RES / "tables" / "split_summary.csv", index=False)
display(split_tab)
log.info("Splits disjoint and hospital-consistent for all protocols")

## 3. QC: implausible values set to missing

In [ ]:
hourly = hourly.merge(patients[["patient_id"]].reset_index().rename(columns={"index": "pidx"}),
                      on="patient_id", how="left", validate="many_to_one")
hourly = hourly.sort_values(["pidx", "t"]).reset_index(drop=True)
assert hourly["pidx"].notna().all()

qc_log = []
for v in DYNAMIC_VARS:
    x = hourly[v].astype(np.float32)
    n_obs0 = int(x.notna().sum())
    n_fix = 0
    if v == "FiO2" and qc_cfg["fio2_percent_to_fraction"]:
        m = (x > 1) & (x <= 100)
        n_fix = int(m.sum())
        x[m] = x[m] / 100.0
    lo, hi = qc_cfg["ranges"][v]
    bad = (x < lo) | (x > hi)
    x[bad] = np.nan
    hourly[v] = x
    qc_log.append({"variable": v, "unit": qc_cfg["units"][v], "observed_before": n_obs0, "fio2_converted": n_fix,
                   "set_missing_implausible": int(bad.sum()), "observed_after": int(x.notna().sum())})
qc_log = pd.DataFrame(qc_log)
qc_log.to_csv(RES / "tables" / "qc_applied.csv", index=False)
display(qc_log[qc_log.set_missing_implausible > 0])

## 4. Causal arrays: values, mask, LOCF values, hours since last observation

In [ ]:
with StageTimer("arrays", log):
    V = hourly[DYNAMIC_VARS].to_numpy(np.float32)                 # raw QC'd, NaN = missing
    M = (~np.isnan(V)).astype(np.uint8)
    pidx = hourly["pidx"].to_numpy(np.int64)
    t = hourly["t"].to_numpy(np.int64)
    # patient offsets (rows are contiguous per patient, ordered by t)
    starts = np.flatnonzero(np.r_[True, pidx[1:] != pidx[:-1]])
    lengths = np.diff(np.r_[starts, len(pidx)])
    assert len(starts) == len(patients) and (pidx[starts] == np.arange(len(patients))).all()
    assert (lengths == patients["n_rows"].values).all()
    # causal LOCF + age via groupby ffill (never bfill)
    dfv = pd.DataFrame(V, columns=DYNAMIC_VARS)
    dfv["pidx"] = pidx
    F = dfv.groupby("pidx")[DYNAMIC_VARS].ffill().to_numpy(np.float32)
    last_t = pd.DataFrame(np.where(M == 1, t[:, None], np.nan), columns=DYNAMIC_VARS)
    last_t["pidx"] = pidx
    last_t = last_t.groupby("pidx")[DYNAMIC_VARS].ffill().to_numpy(np.float32)
    AGE = (t[:, None] - last_t).astype(np.float32)               # NaN before first observation
    AGE = np.minimum(AGE, CFG["max_age_h"])

# Causality self-test: LOCF at row r equals the last observed value at or before r within the same patient
rng = np.random.default_rng(0)
for r in rng.choice(len(V), size=min(2000, len(V)), replace=False):
    s = starts[pidx[r]]
    for j in rng.choice(34, 3, replace=False):
        hist = V[s:r + 1, j]
        ok = np.where(~np.isnan(hist))[0]
        exp = hist[ok[-1]] if len(ok) else np.nan
        assert (np.isnan(exp) and np.isnan(F[r, j])) or np.isclose(exp, F[r, j]), (r, j)
        exp_age = (r - s - ok[-1]) if len(ok) else np.nan
        assert (np.isnan(exp_age) and np.isnan(AGE[r, j])) or np.isclose(min(exp_age, CFG["max_age_h"]), AGE[r, j])
log.info("Causal LOCF / age self-test passed")

## 5. Train-only normalisation statistics (per protocol)

In [ ]:
def fit_norm(train_rows):
    stats = {}
    for j, v in enumerate(DYNAMIC_VARS):
        x = V[train_rows, j]
        x = x[~np.isnan(x)]
        use_log = v in CFG["log1p_vars"]
        if use_log:
            x = np.log1p(np.clip(x, 0, None))
        if len(x) < 2:
            stats[v] = {"log1p": use_log, "lo": 0.0, "hi": 0.0, "mean": 0.0, "std": 1.0, "n": int(len(x))}
            continue
        lo, hi = np.percentile(x, CFG["winsor_pct"])
        xc = np.clip(x, lo, hi)
        sd = float(xc.std())
        stats[v] = {"log1p": use_log, "lo": float(lo), "hi": float(hi), "mean": float(xc.mean()),
                    "std": sd if sd > 1e-6 else 1.0, "n": int(len(x))}
    age = patients.loc[np.unique(pidx[train_rows]), "age"].to_numpy(float)
    icu = np.log1p(hourly.loc[train_rows, "ICULOS"].to_numpy(float).clip(min=0))
    stats["_static"] = {"Age": {"mean": float(np.nanmean(age)), "std": float(np.nanstd(age) or 1.0)},
                        "log_ICULOS": {"mean": float(np.nanmean(icu)), "std": float(np.nanstd(icu) or 1.0)}}
    return stats

def normalize_values(X, stats):
    """Reference numpy implementation (torch version in later notebooks must match it)."""
    out = np.empty_like(X, dtype=np.float32)
    for j, v in enumerate(DYNAMIC_VARS):
        s = stats[v]
        x = X[:, j].astype(np.float64)
        if s["log1p"]:
            x = np.log1p(np.clip(x, 0, None))
        out[:, j] = ((np.clip(x, s["lo"], s["hi"]) - s["mean"]) / s["std"]).astype(np.float32)
    return out

norm = {}
for proto in PROTOCOLS:
    train_p = patients.index[patients[f"split_{proto}"] == "train"].values
    train_rows = np.isin(pidx, train_p)
    norm[proto] = fit_norm(train_rows)
    save_json(norm[proto], ART / f"norm_stats_{proto}.json")
    save_json(norm[proto], RES / "configs" / f"norm_stats_{proto}.json")
    # guard: recompute must not change if test rows are perturbed (statistics depend on train rows only)
    test_rows = np.isin(pidx, patients.index[patients[f"split_{proto}"] == "test"].values)
    V_backup = V[test_rows].copy()
    V[test_rows] = V[test_rows] * 10 + 7
    assert fit_norm(train_rows) == norm[proto], "normalisation depends on test rows!"
    V[test_rows] = V_backup
log.info("Train-only normalisation fitted and leakage-tested for %s", PROTOCOLS)

## 6. Constraint statistics for the physiology loss (training patients only)
For each accepted constraint, the median offset and robust scale of its residual are estimated on **training** patients of each protocol.
`07` uses them to make the physiology loss unit-free: `((residual − offset) / scale)`, with a tolerance band.

In [ ]:
VIDX = {v: j for j, v in enumerate(DYNAMIC_VARS)}
RESIDUALS = {   # identical relations to notebook 02 (raw units)
    "C1_MAP_formula": lambda X: X[:, VIDX["MAP"]] - (X[:, VIDX["SBP"]] + 2 * X[:, VIDX["DBP"]]) / 3,
    "C2_BP_ordering": lambda X: np.maximum(X[:, VIDX["MAP"]] - X[:, VIDX["SBP"]], X[:, VIDX["DBP"]] - X[:, VIDX["MAP"]]),
    "C3_Henderson_Hasselbalch": lambda X: X[:, VIDX["pH"]] - (6.1 + np.log10(X[:, VIDX["HCO3"]] / (0.03 * X[:, VIDX["PaCO2"]]))),
    "C4_Hct_3xHgb": lambda X: X[:, VIDX["Hct"]] - 3 * X[:, VIDX["Hgb"]],
    "C5_bilirubin_direct_le_total": lambda X: X[:, VIDX["Bilirubin_direct"]] - X[:, VIDX["Bilirubin_total"]],
    "C6_SaO2_vs_SpO2": lambda X: X[:, VIDX["SaO2"]] - X[:, VIDX["O2Sat"]],
}
assert set(RESIDUALS) == set(cons["definitions"]), "constraint sets of 02 and 03 differ"
cons_stats = {}
for proto in PROTOCOLS:
    train_rows = np.isin(pidx, patients.index[patients[f"split_{proto}"] == "train"].values)
    Xtr = V[train_rows]
    cons_stats[proto] = {}
    for cid in cons["accepted"]:
        d = cons["definitions"][cid]
        cols = [VIDX[v] for v in d["vars"]]
        ok = ~np.isnan(Xtr[:, cols]).any(axis=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            r = RESIDUALS[cid](Xtr[ok])
        r = r[np.isfinite(r)]
        med = float(np.median(r)) if len(r) else 0.0
        mad = float(1.4826 * np.median(np.abs(r - med))) if len(r) else 1.0
        cons_stats[proto][cid] = {"vars": d["vars"], "kind": d["kind"], "tol": d["tol"], "unit": d["unit"],
                                  "n_train": int(len(r)), "offset": med if d["kind"] == "eq" else 0.0,
                                  "scale": mad if mad > 1e-6 else 1.0}
save_json({"accepted": cons["accepted"], "per_protocol": cons_stats}, ART / "physiology_constraints.json")
save_json({"accepted": cons["accepted"], "per_protocol": cons_stats}, RES / "configs" / "physiology_constraints.json")
save_json(ORGAN_SYSTEMS, ART / "organ_systems.json")
save_json(ORGAN_SYSTEMS, RES / "configs" / "organ_systems.json")
display(pd.DataFrame({p: {c: f"off={s['offset']:.3g}, scale={s['scale']:.3g}, n={s['n_train']}"
                          for c, s in d.items()} for p, d in cons_stats.items()}))

## 7. Save arrays and tables

In [ ]:
with StageTimer("save", log):
    np.save(ART / "X_values.npy", V)
    np.save(ART / "X_mask.npy", M)
    np.save(ART / "X_locf.npy", F.astype(np.float32))
    np.save(ART / "X_age.npy", AGE.astype(np.float16))
    np.save(ART / "patient_starts.npy", starts.astype(np.int64))
    np.save(ART / "patient_lengths.npy", lengths.astype(np.int64))
    rows_meta = pd.DataFrame({"pidx": pidx.astype(np.int32), "t": t.astype(np.int32),
                              "ICULOS": hourly["ICULOS"].to_numpy(np.float32),
                              "SepsisLabel": hourly["SepsisLabel"].to_numpy(np.int8)})
    save_table(rows_meta, ART / "rows_meta.parquet")
    save_table(patients, ART / "patients_split.parquet")
save_json({"config": CFG, "dynamic_vars": DYNAMIC_VARS, "n_rows": int(len(V)), "n_patients": int(len(patients)),
           "arrays": {"X_values": "float32 raw QC'd, NaN=missing", "X_mask": "uint8 1=observed",
                      "X_locf": "float32 causal LOCF, NaN before first obs",
                      "X_age": f"float16 hours since last obs (cap {CFG['max_age_h']}), NaN before first obs"}},
          ART / "preprocessing_config.json")
save_json(CFG, RES / "configs" / "preprocessing_config.json")
print("✅ Preprocessing complete. Next: 04_temporal_windowing (attach outputs of 01–03).")

# 04 · Causal temporal windows, targets and label-efficiency subsets
**Inputs:** outputs of `01`–`03`.

**Design**
- A sample is one **(patient, hour r)** prediction time. Its input is the last `W` hours `[r−W+1, r]` of the same patient, left-padded.
  A *carry-in* vector (the LOCF value and its age just before the window) keeps information about labs measured before the window, with no look-ahead.
- Windows are built on the fly from the arrays of `03`, so nothing is duplicated W times on disk. The builder is written once to `ptcssl_data.py` and imported by every later notebook (single source of truth).
- **Targets**
  - `y_official = SepsisLabel` (already shifted: 1 for `t ≥ t_sepsis − 6`). Used for training and for the official utility.
  - `hours_to_onset = t_sepsis − t`, and horizon targets `y_h = 1 if 0 ≤ hours_to_onset ≤ h` for h ∈ {4, 6, 8, 12}. Post-onset rows are excluded from the horizon analysis.
    Self-check: `y_6` must equal `SepsisLabel` on every pre-onset row. This confirms the 6-hour label shift is handled correctly.
- **Label-efficiency subsets** (5, 10, 25, 50, 100% of training patients; 5 seeds): stratified by hospital × septic, patient-level, **nested** (5% ⊂ 10% ⊂ …) for each seed.
  By default the validation set is subsampled to the same fraction, so model selection does not use more labels than the budget allows.
- **Leakage tests:** future rows are perturbed and every window must stay identical; windows never mix patients; subsets stay inside their split.

CPU, about 1–3 min.

In [ ]:
# ===== PTC-SSL common setup (identical in every notebook; do not edit per notebook) =====
import os, sys, json, time, math, glob, shutil, random, hashlib, logging, platform, warnings
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Kaggle paths (overridable through environment variables for local testing only)
WORK = Path(os.environ.get("PTCSSL_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("PTCSSL_INPUT", "/kaggle/input"))
ART = WORK / "artifacts"   # data artifacts consumed by later notebooks (may contain patient-level data)
RES = WORK / "results"     # metrics / tables / figures / configs (no raw patient data) -> paper package
LOGS = WORK / "logs"
for _d in (ART, RES, LOGS, RES / "figures", RES / "tables", RES / "configs"):
    _d.mkdir(parents=True, exist_ok=True)

GLOBAL_SEED = 2026

def set_seed(seed: int = GLOBAL_SEED):
    """Seed python, numpy and (if present) torch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed()

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
        fh = logging.FileHandler(LOGS / f"{name}.log", mode="a")
        fh.setFormatter(fmt)
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(fh)
        logger.addHandler(sh)
        logger.propagate = False
    return logger

try:
    display  # noqa: F821  (exists inside Jupyter/Kaggle)
except NameError:
    def display(x):
        print(x)

def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def md5_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    return path

def load_json(path):
    with open(path) as f:
        return json.load(f)

def save_table(df: pd.DataFrame, path):
    """Save as parquet (Kaggle has pyarrow); fall back to pickle if parquet is unavailable."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except ImportError:
        alt = path.with_suffix(".pkl")
        df.to_pickle(alt)
        return alt

def load_table(path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix == ".parquet" and path.exists():
        return pd.read_parquet(path)
    alt = path.with_suffix(".pkl")
    if alt.exists():
        return pd.read_pickle(alt)
    if path.exists():
        return pd.read_pickle(path)
    raise FileNotFoundError(path)

_ARTIFACT_CACHE = {}

def find_artifact(name: str, producer: str) -> Path:
    """Locate an artifact produced by an earlier notebook.
    Search order: this session's /kaggle/working/artifacts, then any attached input (/kaggle/input/**).
    Table artifacts may exist as .parquet or .pkl."""
    if name in _ARTIFACT_CACHE:
        return _ARTIFACT_CACHE[name]
    stem, suffix = os.path.splitext(name)
    names = [name] + ([stem + ".pkl"] if suffix == ".parquet" else [])
    for n in names:
        p = ART / n
        if p.exists():
            _ARTIFACT_CACHE[name] = p
            return p
    hits = []
    if INPUT.exists():
        for dirpath, _, files in os.walk(INPUT, followlinks=True):
            if Path(dirpath).name == "artifacts":          # never pick up copies under results/
                hits += [Path(dirpath) / n for n in names if n in files]
    if hits:
        hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        if len(hits) > 1:
            print(f"[find_artifact] {len(hits)} copies of {name} found; using newest: {hits[0]}")
        _ARTIFACT_CACHE[name] = hits[0]
        return hits[0]
    raise FileNotFoundError(
        f"Artifact '{name}' not found. Run notebook {producer} first, then attach its output to this "
        f"notebook (Kaggle: Add Input -> Your Work -> Notebooks -> {producer})."
    )

def env_info() -> dict:
    info = {
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
    for mod in ("scipy", "sklearn", "matplotlib", "torch", "lightgbm"):
        try:
            m = __import__(mod)
            info[mod] = getattr(m, "__version__", "unknown")
        except ImportError:
            info[mod] = "not installed"
    try:
        import torch
        info["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return info

class StageTimer:
    """Context manager that logs wall-clock time of a stage (also used for the compute-cost table)."""
    def __init__(self, name, logger=None):
        self.name, self.logger = name, logger
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        self.seconds = time.time() - self.t0
        msg = f"[time] {self.name}: {self.seconds:.1f}s"
        (self.logger.info if self.logger else print)(msg)

# PhysioNet 2019 schema (verified against the official data description, physionet.org/content/challenge-2019/1.0.0/)
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos", "Calcium",
        "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate", "Magnesium", "Phosphate",
        "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets"]
DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
DYNAMIC_VARS = VITALS + LABS                       # 34 time-varying clinical variables
EXPECTED_COLUMNS = DYNAMIC_VARS + DEMOGRAPHICS + ["SepsisLabel"]   # 41 columns, official order

# Official utility parameters (physionetchallenges/evaluation-2019, evaluate_sepsis_score.py)
UTILITY_PARAMS = dict(dt_early=-12, dt_optimal=-6, dt_late=3, max_u_tp=1.0, min_u_fn=-2.0, u_fp=-0.05, u_tn=0.0)
print(f"[setup] WORK={WORK}  INPUT={INPUT}  seed={GLOBAL_SEED}")

In [ ]:
NB = "04_temporal_windowing"
log = get_logger(NB)
CFG = {
    "window_hours": int(os.environ.get("PTCSSL_W", 24)),
    "horizons_h": [4, 6, 8, 12],
    "label_fractions": [0.05, 0.10, 0.25, 0.50, 1.00],
    "subset_seeds": [0, 1, 2, 3, 4],
    "val_follows_label_fraction": True,
    "min_septic_per_subset": 1,
}
PROTOCOLS = ["pooled", "A2B", "B2A"]
V = np.load(find_artifact("X_values.npy", "03_preprocessing"), mmap_mode="r")
M = np.load(find_artifact("X_mask.npy", "03_preprocessing"), mmap_mode="r")
F = np.load(find_artifact("X_locf.npy", "03_preprocessing"), mmap_mode="r")
AGE = np.load(find_artifact("X_age.npy", "03_preprocessing"), mmap_mode="r")
starts = np.load(find_artifact("patient_starts.npy", "03_preprocessing"))
lengths = np.load(find_artifact("patient_lengths.npy", "03_preprocessing"))
rows_meta = load_table(find_artifact("rows_meta.parquet", "03_preprocessing"))
patients = load_table(find_artifact("patients_split.parquet", "03_preprocessing"))
assert V.shape == M.shape == F.shape == AGE.shape and V.shape[1] == 34 and len(rows_meta) == len(V)
log.info("Arrays: %s rows, %d patients, W=%d", V.shape[0], len(patients), CFG["window_hours"])

## 1. Shared data module `ptcssl_data.py` (window builder used by notebooks 05–11)

In [ ]:
PTCSSL_DATA_SRC = r'''
"""ptcssl_data.py - causal window construction for PTC-SSL (written by notebook 04; do not edit by hand)."""
import numpy as np

class WindowBuilder:
    """Builds causal windows ending at global row r (inclusive), left-padded to W hours.

    Returns dict of numpy arrays for a batch of rows R:
      values [B,W,34] raw QC'd values (NaN where missing or padding)
      mask   [B,W,34] 1 = observed
      pad    [B,W]    1 = padding (before the patient's first row)
      carry_val [B,34] LOCF value just before the window (NaN if none)
      carry_age [B,34] hours since that observation, measured at the window's first hour (NaN if none)
      pidx   [B]      patient index
    Only rows <= r of the same patient are ever read (verified by notebook 04)."""

    def __init__(self, V, M, F, AGE, starts, row_pidx, W):
        self.V, self.M, self.F, self.AGE = V, M, F, AGE
        self.starts = np.asarray(starts)
        self.row_pidx = np.asarray(row_pidx)
        self.W = int(W)
        self.offsets = np.arange(-self.W + 1, 1)

    def get(self, R):
        R = np.asarray(R, dtype=np.int64)
        p = self.row_pidx[R]
        s0 = self.starts[p]
        idx = R[:, None] + self.offsets[None, :]                  # [B,W]
        valid = idx >= s0[:, None]
        idx_c = np.where(valid, idx, R[:, None])                   # any in-patient row; masked out below
        values = np.asarray(self.V[idx_c.ravel()]).reshape(len(R), self.W, -1).astype(np.float32)
        mask = np.asarray(self.M[idx_c.ravel()]).reshape(len(R), self.W, -1).astype(np.float32)
        values[~valid] = np.nan
        mask[~valid] = 0.0
        first = np.maximum(s0, R - self.W + 1)
        has_carry = first > s0
        prev = np.where(has_carry, first - 1, R)
        carry_val = np.asarray(self.F[prev]).astype(np.float32)
        carry_age = np.asarray(self.AGE[prev]).astype(np.float32) + 1.0
        carry_val[~has_carry] = np.nan
        carry_age[~has_carry] = np.nan
        return {"values": values, "mask": mask, "pad": (~valid).astype(np.float32),
                "carry_val": carry_val, "carry_age": carry_age, "pidx": p}

def normalize_np(X, stats, var_names):
    """Same transform as notebook 03: optional log1p -> winsorise -> z-score. NaN stays NaN."""
    out = np.empty(X.shape, dtype=np.float32)
    for j, v in enumerate(var_names):
        s = stats[v]
        x = X[..., j].astype(np.float64)
        if s["log1p"]:
            x = np.log1p(np.clip(x, 0, None))
        out[..., j] = (np.clip(x, s["lo"], s["hi"]) - s["mean"]) / s["std"]
    return out

def static_features(age, gender, iculos, stats):
    a = (np.asarray(age, float) - stats["_static"]["Age"]["mean"]) / stats["_static"]["Age"]["std"]
    l = (np.log1p(np.clip(np.asarray(iculos, float), 0, None)) - stats["_static"]["log_ICULOS"]["mean"]) \
        / stats["_static"]["log_ICULOS"]["std"]
    return np.stack([np.nan_to_num(a), np.nan_to_num(np.asarray(gender, float), nan=0.5), np.nan_to_num(l)], -1).astype(np.float32)
'''
(ART / "ptcssl_data.py").write_text(PTCSSL_DATA_SRC)
sys.path.insert(0, str(ART))
import importlib
import ptcssl_data
importlib.reload(ptcssl_data)
from ptcssl_data import WindowBuilder, normalize_np, static_features

row_pidx = rows_meta["pidx"].to_numpy(np.int64)
wb = WindowBuilder(V, M, F, AGE, starts, row_pidx, CFG["window_hours"])

## 2. Leakage and correctness tests for the window builder

In [ ]:
rng = np.random.default_rng(GLOBAL_SEED)
test_rows = rng.choice(len(V), size=min(3000, len(V)), replace=False)
base = wb.get(test_rows)

# (a) windows never mix patients and padding is exactly the pre-admission part
W = CFG["window_hours"]
for k, r in enumerate(test_rows[:500]):
    p = row_pidx[r]
    n_hist = r - starts[p] + 1
    assert base["pad"][k].sum() == max(0, W - n_hist)
    lo = max(starts[p], r - W + 1)
    assert np.array_equal(np.nan_to_num(base["values"][k, W - (r - lo + 1):], nan=-999),
                          np.nan_to_num(np.asarray(V[lo:r + 1]), nan=-999))

# (b) causality: perturb every row strictly after r (same patient) -> window must be identical
Vc, Mc, Fc, Ac = (np.array(a) for a in (V, M, F, AGE))
wb_c = WindowBuilder(Vc, Mc, Fc, Ac, starts, row_pidx, W)
for r in test_rows[:300]:
    p = row_pidx[r]
    seg = slice(r + 1, starts[p] + lengths[p])          # strictly future rows of the same patient
    backup = [a[seg].copy() for a in (Vc, Mc, Fc, Ac)]
    Vc[seg] += 1000.0; Mc[seg] = 1 - Mc[seg]; Fc[seg] += 1000.0; Ac[seg] += 5.0
    a, b = wb.get([r]), wb_c.get([r])
    for key in ("values", "mask", "carry_val", "carry_age"):
        assert np.array_equal(np.nan_to_num(a[key], nan=-1), np.nan_to_num(b[key], nan=-1)), (r, key)
    Vc[seg], Mc[seg], Fc[seg], Ac[seg] = backup
del Vc, Mc, Fc, Ac
# (c) carry-in equals the latest observation before the window
for k, r in enumerate(test_rows[:300]):
    p = row_pidx[r]
    first = max(starts[p], r - W + 1)
    if first > starts[p]:
        hist = np.asarray(V[starts[p]:first])
        for j in range(34):
            o = np.where(~np.isnan(hist[:, j]))[0]
            exp = hist[o[-1], j] if len(o) else np.nan
            got = base["carry_val"][k, j]
            assert (np.isnan(exp) and np.isnan(got)) or np.isclose(exp, got)
log.info("Window builder tests passed: no patient mixing, exact padding, strict causality, correct carry-in")

## 3. Targets: official label, hours to onset, horizon labels (with the 6-hour shift check)

In [ ]:
assert (patients.index == np.arange(len(patients))).all()   # pidx == positional index
pt = patients
t_sep = pt["t_sepsis_idx"].to_numpy()[row_pidx].astype(float)
t_sep[pt["septic"].to_numpy()[row_pidx] == 0] = np.nan
t_row = rows_meta["t"].to_numpy(float)
y_off = rows_meta["SepsisLabel"].to_numpy(np.int8)
hto = t_sep - t_row                           # hours to onset (NaN for non-septic)
targets = pd.DataFrame({"y_official": y_off, "hours_to_onset": hto.astype(np.float32)})
for h in CFG["horizons_h"]:
    y = np.zeros(len(t_row), dtype=np.float32)
    sep = ~np.isnan(hto)
    y[sep] = ((hto[sep] >= 0) & (hto[sep] <= h)).astype(np.float32)
    y[sep & (hto < 0)] = np.nan               # post-onset rows excluded from horizon analysis
    targets[f"y_h{h}"] = y

pre = ~np.isnan(hto) & (hto >= 0)
assert np.array_equal(targets.loc[pre, "y_h6"].to_numpy(), y_off[pre].astype(np.float32)), \
    "6-hour shift check failed: y_h6 must equal SepsisLabel before onset"
assert (y_off[np.isnan(hto)] == 0).all(), "non-septic patients must have SepsisLabel 0"
log.info("Label-shift check passed: SepsisLabel == 1[0 <= t_sepsis - t <= 6] on all pre-onset rows")
save_table(targets, ART / "targets.parquet")

## 4. Sample indices per protocol and split

In [ ]:
idx_summary = []
for proto in PROTOCOLS:
    split_of_row = patients[f"split_{proto}"].to_numpy()[row_pidx]
    for sp in ("train", "val", "test"):
        R = np.flatnonzero(split_of_row == sp).astype(np.int64)
        np.save(ART / f"idx_{proto}_{sp}.npy", R)
        idx_summary.append({"protocol": proto, "split": sp, "samples": len(R),
                            "patients": int(len(np.unique(row_pidx[R]))),
                            "positive_rate_official": float(y_off[R].mean()) if len(R) else np.nan})
idx_summary = pd.DataFrame(idx_summary)
idx_summary.to_csv(RES / "tables" / "sample_counts.csv", index=False)
display(idx_summary)

## 5. Nested, stratified label-efficiency subsets (patient level)

In [ ]:
def nested_subsets(pids, strata, fracs, seed, min_pos):
    """Return {frac: set(pids)} with nesting guaranteed (same permutation, growing prefix per stratum)."""
    rng = np.random.default_rng(seed)
    out = {f: [] for f in fracs}
    for s in np.unique(strata):
        members = pids[strata == s].copy()
        rng.shuffle(members)
        is_pos = s.endswith("_1")
        for f in fracs:
            k = int(np.ceil(f * len(members) - 1e-9))
            if is_pos:
                k = max(k, min(min_pos, len(members)))
            out[f].extend(members[:k].tolist())
    return {f: set(v) for f, v in out.items()}

sub_rows, sub_tables = [], {}
for proto in PROTOCOLS:
    tab = patients[["patient_id", "hospital", "septic", f"split_{proto}"]].copy()
    for sp in ("train", "val"):
        part = tab[tab[f"split_{proto}"] == sp]
        pids = part.index.to_numpy()
        strata = (part["hospital"] + "_" + part["septic"].astype(str)).to_numpy()
        for seed in CFG["subset_seeds"]:
            subs = nested_subsets(pids, strata, CFG["label_fractions"], 10_000 + seed, CFG["min_septic_per_subset"])
            fr = CFG["label_fractions"]
            for a, b in zip(fr[:-1], fr[1:]):
                assert subs[a] <= subs[b], "subsets must be nested"
            assert subs[1.0] == set(pids)
            for f in fr:
                col = f"{sp}_s{seed}_f{int(round(f * 100)):03d}"
                tab[col] = tab.index.isin(list(subs[f]))
                chosen = tab.loc[tab[col]]
                sub_rows.append({"protocol": proto, "split": sp, "seed": seed, "fraction": f,
                                 "patients": len(chosen), "septic_patients": int(chosen.septic.sum()),
                                 "labelled_hours": int(np.isin(row_pidx, chosen.index.to_numpy()).sum())})
    # no subset may contain patients from another split
    for c in [c for c in tab.columns if c.startswith(("train_s", "val_s"))]:
        sp = c.split("_")[0]
        assert (tab.loc[tab[c], f"split_{proto}"] == sp).all()
    sub_tables[proto] = tab
    save_table(tab.reset_index().rename(columns={"index": "pidx"}), ART / f"label_subsets_{proto}.parquet")
subs_df = pd.DataFrame(sub_rows)
subs_df.to_csv(RES / "tables" / "label_subset_sizes.csv", index=False)
display(subs_df[(subs_df.seed == 0)].pivot_table(index=["protocol", "split"], columns="fraction",
                                                  values="septic_patients"))

## 6. Throughput benchmark and configuration

In [ ]:
stats_pooled = load_json(find_artifact("norm_stats_pooled.json", "03_preprocessing"))
R = np.load(ART / "idx_pooled_train.npy")[:4096]
t0 = time.time()
batch = wb.get(R)
xn = normalize_np(batch["values"], stats_pooled, DYNAMIC_VARS)
el = time.time() - t0
assert np.isnan(xn).sum() == np.isnan(batch["values"]).sum(), "normalisation must not create/remove NaN"
log.info("Built + normalised %d windows in %.3fs (%.0f windows/s)", len(R), el, len(R) / max(el, 1e-9))

window_cfg = {**CFG, "n_dynamic": 34, "dynamic_vars": DYNAMIC_VARS, "static_features": ["Age", "Gender", "log_ICULOS"],
              "carry_in": True, "builder_module": "artifacts/ptcssl_data.py",
              "builder_sha256": sha256_file(ART / "ptcssl_data.py"),
              "targets": ["y_official"] + [f"y_h{h}" for h in CFG["horizons_h"]]}
save_json(window_cfg, ART / "window_config.json")
save_json(window_cfg, RES / "configs" / "window_config.json")
print("✅ Windowing complete. Next: 05_supervised_baselines (attach outputs of 01–04; GPU recommended).")

# 06 · Full experiments: 5 seeds, ablations, external validation, multi-seed statistics
Runs after 01–04 in the same session (or with their outputs attached).

**Jobs (fixed before seeing results; same encoder, inputs, budget and hyper-parameters as notebook 05)**
1. **Main**: pooled protocol, arms LightGBM / supervised GRU / generic SSL / PTC-SSL, label fractions 5, 10, 25, 100%, seeds 0–4.
2. **Ablations** (pooled, 5% and 10% labels, seeds 0–4): PTC-SSL without organ masking (random masking), without L_phys, without L_temp, values-only input (mask and time-since-observation channels zeroed), missingness-only input (value channels zeroed).
3. **External validation**: train/val on one hospital, test on the other (A→B, B→A), 10% and 100% labels, seeds 0–2.
4. **Summary**: mean ± SD over seeds; paired patient-level bootstrap on seed-ensembled predictions; per-seed win counts; calibration (ECE, reliability bins); figures; result package.

A smoke test runs every arm on a few rows first, so a bug fails in minutes, not hours. A time guard skips the remaining runs (never the summary) if the budget is exceeded. Every run is appended to `results/tables/runs_metrics.csv` and skipped on resume.

In [ ]:
# ===== PTC-SSL common setup (identical in every notebook; do not edit per notebook) =====
import os, sys, json, time, math, glob, shutil, random, hashlib, logging, platform, warnings
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Kaggle paths (overridable through environment variables for local testing only)
WORK = Path(os.environ.get("PTCSSL_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("PTCSSL_INPUT", "/kaggle/input"))
ART = WORK / "artifacts"   # data artifacts consumed by later notebooks (may contain patient-level data)
RES = WORK / "results"     # metrics / tables / figures / configs (no raw patient data) -> paper package
LOGS = WORK / "logs"
for _d in (ART, RES, LOGS, RES / "figures", RES / "tables", RES / "configs"):
    _d.mkdir(parents=True, exist_ok=True)

GLOBAL_SEED = 2026

def set_seed(seed: int = GLOBAL_SEED):
    """Seed python, numpy and (if present) torch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed()

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
        fh = logging.FileHandler(LOGS / f"{name}.log", mode="a")
        fh.setFormatter(fmt)
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(fh)
        logger.addHandler(sh)
        logger.propagate = False
    return logger

try:
    display  # noqa: F821  (exists inside Jupyter/Kaggle)
except NameError:
    def display(x):
        print(x)

def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def md5_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    return path

def load_json(path):
    with open(path) as f:
        return json.load(f)

def save_table(df: pd.DataFrame, path):
    """Save as parquet (Kaggle has pyarrow); fall back to pickle if parquet is unavailable."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except ImportError:
        alt = path.with_suffix(".pkl")
        df.to_pickle(alt)
        return alt

def load_table(path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix == ".parquet" and path.exists():
        return pd.read_parquet(path)
    alt = path.with_suffix(".pkl")
    if alt.exists():
        return pd.read_pickle(alt)
    if path.exists():
        return pd.read_pickle(path)
    raise FileNotFoundError(path)

_ARTIFACT_CACHE = {}

def find_artifact(name: str, producer: str) -> Path:
    """Locate an artifact produced by an earlier notebook.
    Search order: this session's /kaggle/working/artifacts, then any attached input (/kaggle/input/**).
    Table artifacts may exist as .parquet or .pkl."""
    if name in _ARTIFACT_CACHE:
        return _ARTIFACT_CACHE[name]
    stem, suffix = os.path.splitext(name)
    names = [name] + ([stem + ".pkl"] if suffix == ".parquet" else [])
    for n in names:
        p = ART / n
        if p.exists():
            _ARTIFACT_CACHE[name] = p
            return p
    hits = []
    if INPUT.exists():
        for dirpath, _, files in os.walk(INPUT, followlinks=True):
            if Path(dirpath).name == "artifacts":          # never pick up copies under results/
                hits += [Path(dirpath) / n for n in names if n in files]
    if hits:
        hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        if len(hits) > 1:
            print(f"[find_artifact] {len(hits)} copies of {name} found; using newest: {hits[0]}")
        _ARTIFACT_CACHE[name] = hits[0]
        return hits[0]
    raise FileNotFoundError(
        f"Artifact '{name}' not found. Run notebook {producer} first, then attach its output to this "
        f"notebook (Kaggle: Add Input -> Your Work -> Notebooks -> {producer})."
    )

def env_info() -> dict:
    info = {
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
    for mod in ("scipy", "sklearn", "matplotlib", "torch", "lightgbm"):
        try:
            m = __import__(mod)
            info[mod] = getattr(m, "__version__", "unknown")
        except ImportError:
            info[mod] = "not installed"
    try:
        import torch
        info["cuda_available"] = bool(torch.cuda.is_available())
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return info

class StageTimer:
    """Context manager that logs wall-clock time of a stage (also used for the compute-cost table)."""
    def __init__(self, name, logger=None):
        self.name, self.logger = name, logger
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        self.seconds = time.time() - self.t0
        msg = f"[time] {self.name}: {self.seconds:.1f}s"
        (self.logger.info if self.logger else print)(msg)

# PhysioNet 2019 schema (verified against the official data description, physionet.org/content/challenge-2019/1.0.0/)
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos", "Calcium",
        "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate", "Magnesium", "Phosphate",
        "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets"]
DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
DYNAMIC_VARS = VITALS + LABS                       # 34 time-varying clinical variables
EXPECTED_COLUMNS = DYNAMIC_VARS + DEMOGRAPHICS + ["SepsisLabel"]   # 41 columns, official order

# Official utility parameters (physionetchallenges/evaluation-2019, evaluate_sepsis_score.py)
UTILITY_PARAMS = dict(dt_early=-12, dt_optimal=-6, dt_late=3, max_u_tp=1.0, min_u_fn=-2.0, u_fp=-0.05, u_tn=0.0)
print(f"[setup] WORK={WORK}  INPUT={INPUT}  seed={GLOBAL_SEED}")

In [ ]:
NB = "06_full_experiments"
log = get_logger(NB)
import torch
import torch.nn as nn
import torch.nn.functional as TF
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

T_START = time.time()
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, os.cpu_count() or 1))
CFG = {
    "hidden": 96, "batch": 256, "lr": 1e-3, "pos_weight": 10.0,
    "pre_epochs": 4, "pre_windows_per_epoch": 120_000,
    "ft_max_epochs": 8, "ft_rows_per_epoch": 100_000, "patience": 2, "val_rows_cap": 60_000,
    "rand_mask_p": 0.15, "organ_block_h": 8, "lambda": 0.1,
    "max_delta_h": 168.0, "clip_z": 5.0, "bootstrap_B": 200,
    "time_budget_h": float(os.environ.get("PTCSSL_BUDGET_H", 8.0)),
}
MAIN_ARMS = ["lightgbm", "supervised_gru", "generic_ssl", "ptc_ssl"]
ABL_ARMS = ["abl_no_organ", "abl_no_phys", "abl_no_temp", "abl_values_only", "abl_missing_only"]
JOBS = [
    {"name": "main", "protocol": "pooled", "seeds": [0, 1, 2, 3, 4], "fractions": [0.05, 0.10, 0.25, 1.00], "arms": MAIN_ARMS},
    {"name": "ablation", "protocol": "pooled", "seeds": [0, 1, 2, 3, 4], "fractions": [0.05, 0.10], "arms": ABL_ARMS},
    {"name": "external", "protocol": "A2B", "seeds": [0, 1, 2], "fractions": [0.10, 1.00], "arms": MAIN_ARMS},
    {"name": "external", "protocol": "B2A", "seeds": [0, 1, 2], "fractions": [0.10, 1.00], "arms": MAIN_ARMS},
]
# SSL variants: (masking, lambda_phys, lambda_temp, input mode)
L = CFG["lambda"]
VARIANTS = {"generic_ssl": ("random", 0.0, 0.0, "full"), "ptc_ssl": ("organ", L, L, "full"),
            "abl_no_organ": ("random", L, L, "full"), "abl_no_phys": ("organ", 0.0, L, "full"),
            "abl_no_temp": ("organ", L, 0.0, "full"), "abl_values_only": ("organ", L, L, "values"),
            "abl_missing_only": ("organ", L, L, "missing")}
if os.environ.get("PTCSSL_QUICK"):
    CFG.update(pre_epochs=1, pre_windows_per_epoch=2000, ft_max_epochs=1, ft_rows_per_epoch=2000, val_rows_cap=2000, bootstrap_B=20)
    for j in JOBS:
        j["seeds"] = j["seeds"][:2]
log.info("Device=%s threads=%d CFG=%s", DEV, torch.get_num_threads(), CFG)
if DEV.type == "cuda":
    log.info("GPU: %s", torch.cuda.get_device_name(0))
save_json({"CFG": CFG, "JOBS": JOBS, "VARIANTS": VARIANTS}, RES / "configs" / "full_experiment_config.json")

In [ ]:
# ---- protocol-independent artifacts ----
sys.path.insert(0, str(find_artifact("ptcssl_data.py", "04_temporal_windowing").parent))
from ptcssl_data import normalize_np
V = np.load(find_artifact("X_values.npy", "03_preprocessing"))
M = np.load(find_artifact("X_mask.npy", "03_preprocessing")).astype(bool)
F = np.load(find_artifact("X_locf.npy", "03_preprocessing"))
AGE = np.load(find_artifact("X_age.npy", "03_preprocessing")).astype(np.float32)
starts = np.load(find_artifact("patient_starts.npy", "03_preprocessing"))
rows_meta = load_table(find_artifact("rows_meta.parquet", "03_preprocessing"))
patients = load_table(find_artifact("patients_split.parquet", "03_preprocessing"))
targets = load_table(find_artifact("targets.parquet", "04_temporal_windowing"))
CONS = load_json(find_artifact("physiology_constraints.json", "03_preprocessing"))
ORGANS = load_json(find_artifact("organ_systems.json", "03_preprocessing"))
W = int(load_json(find_artifact("window_config.json", "04_temporal_windowing"))["window_hours"])
row_pidx = rows_meta["pidx"].to_numpy(np.int64)
row_t = rows_meta["t"].to_numpy()
y_all = targets["y_official"].to_numpy(np.float32)
NV = 34
VIDX = {v: j for j, v in enumerate(DYNAMIC_VARS)}
ORGAN_MASKS = np.stack([np.isin(np.arange(NV), [VIDX[v] for v in vs]) for vs in ORGANS.values()])
st_gen = np.nan_to_num(patients["gender"].to_numpy(float), nan=0.5).astype(np.float32)
CS = np.cumsum(M, axis=0, dtype=np.int32)
INPUT_MODE = "full"

def T(a):
    return torch.as_tensor(a, device=DEV)

def use_protocol(proto):
    """Rebind every protocol-dependent global (normalisation, splits, subsets, constraint stats)."""
    global P, NORM, subsets, IDX, VN, FN, st_age, row_icu, CONS_P, _mu, _sd, _islog
    P = proto
    NORM = load_json(find_artifact(f"norm_stats_{P}.json", "03_preprocessing"))
    subsets = load_table(find_artifact(f"label_subsets_{P}.parquet", "04_temporal_windowing")).set_index("pidx")
    IDX = {sp: np.load(find_artifact(f"idx_{P}_{sp}.npy", "04_temporal_windowing")) for sp in ("train", "val", "test")}
    VN = normalize_np(V, NORM, DYNAMIC_VARS)
    FN = normalize_np(F, NORM, DYNAMIC_VARS)
    st_age = ((patients["age"].to_numpy(float) - NORM["_static"]["Age"]["mean"]) / NORM["_static"]["Age"]["std"]).astype(np.float32)
    row_icu = ((np.log1p(rows_meta["ICULOS"].to_numpy(float).clip(0)) - NORM["_static"]["log_ICULOS"]["mean"])
               / NORM["_static"]["log_ICULOS"]["std"]).astype(np.float32)
    CONS_P = CONS["per_protocol"][P]
    _mu = T(np.array([NORM[v]["mean"] for v in DYNAMIC_VARS], np.float32))
    _sd = T(np.array([NORM[v]["std"] for v in DYNAMIC_VARS], np.float32))
    _islog = T(np.array([NORM[v]["log1p"] for v in DYNAMIC_VARS]))
    for sp, R in IDX.items():
        assert patients.loc[np.unique(row_pidx[R]), f"split_{P}"].eq(sp).all(), "split leakage"
    log.info("protocol %s: rows %s", P, {k: len(v) for k, v in IDX.items()})

use_protocol("pooled")

In [ ]:
OFF = np.arange(-W + 1, 1)
TSTEP = np.arange(W)
LOGMAX = np.log1p(CFG["max_delta_h"])

def build(R, mask_mode=None, rng=None):
    R = np.asarray(R, dtype=np.int64)
    B = len(R)
    p = row_pidx[R]
    s0 = starts[p]
    idx = R[:, None] + OFF
    valid = idx >= s0[:, None]
    idc = np.where(valid, idx, R[:, None])
    m = M[idc] & valid[..., None]
    x = np.where(m, VN[idc], np.nan).astype(np.float32)
    sel = np.zeros_like(m)
    if mask_mode == "random":
        sel = m & (rng.random(m.shape) < CFG["rand_mask_p"])
    elif mask_mode == "organ":
        sys_id = rng.integers(0, len(ORGAN_MASKS), B)
        Lb = CFG["organ_block_h"]
        t0 = rng.integers(0, W - Lb + 1, B)
        tblock = (TSTEP[None, :] >= t0[:, None]) & (TSTEP[None, :] < (t0 + Lb)[:, None])
        sel = m & tblock[..., None] & ORGAN_MASKS[sys_id][:, None, :]
        empty = ~sel.reshape(B, -1).any(1)
        if empty.any():
            sel[empty] = m[empty] & (rng.random(m[empty].shape) < CFG["rand_mask_p"])
    m_in = m & ~sel
    first = np.maximum(s0, R - W + 1)
    has = first > s0
    prev = np.where(has, first - 1, R)
    cv = FN[prev].copy(); cv[~has] = np.nan
    ca = AGE[prev] + 1.0; ca[~has] = np.nan
    last = np.maximum.accumulate(np.where(m_in, TSTEP[None, :, None], -1), axis=1)
    has_in = last >= 0
    ff = np.take_along_axis(np.nan_to_num(np.where(m_in, x, 0.0)), np.maximum(last, 0), axis=1)
    ff = np.where(has_in, ff, np.nan_to_num(cv, nan=0.0)[:, None, :])
    delta = np.where(has_in, TSTEP[None, :, None] - last,
                     np.where(np.isnan(ca)[:, None, :], CFG["max_delta_h"], ca[:, None, :] + TSTEP[None, :, None]))
    delta = np.log1p(np.minimum(delta, CFG["max_delta_h"])) / LOGMAX
    X = np.concatenate([np.clip(ff, -CFG["clip_z"], CFG["clip_z"]), m_in.astype(np.float32),
                        delta.astype(np.float32), (~valid)[..., None].astype(np.float32)], axis=-1).astype(np.float32)
    if INPUT_MODE == "missing":
        X[..., :NV] = 0.0                       # values hidden: only what was measured and when
    elif INPUT_MODE == "values":
        X[..., NV:3 * NV] = 0.0                 # measurement pattern hidden: only (carried-forward) values
    st = np.stack([np.nan_to_num(st_age[p]), st_gen[p], row_icu[R]], -1).astype(np.float32)
    return X, st, x, sel, m_in

# causality spot-check (as in 05)
_rng = np.random.default_rng(0)
_up = np.unique(row_pidx[IDX["train"]])
_pids = _rng.choice(_up, min(50, len(_up)), replace=False)
_R = np.array([_rng.choice(IDX["train"][row_pidx[IDX["train"]] == q]) for q in _pids])
_X1 = build(_R)[0]
_fut = np.zeros(len(VN), bool)
for r in _R:
    q = row_pidx[r]
    stop = starts[q + 1] if q + 1 < len(starts) else len(VN)
    _fut[r + 1:stop] = True
_bkV, _bkF = VN[_fut].copy(), FN[_fut].copy()
VN[_fut] += 100.0; FN[_fut] += 100.0
_X2 = build(_R)[0]
VN[_fut], FN[_fut] = _bkV, _bkF
del _bkV, _bkF
assert np.allclose(_X1, _X2), "batch builder reads future rows!"
IN_DIM = _X1.shape[-1]
log.info("Batch builder causality check passed; input dim=%d", IN_DIM)

In [ ]:
class Net(nn.Module):
    def __init__(self, d_in, h):
        super().__init__()
        self.gru = nn.GRU(d_in, h, batch_first=True)
        self.fuse = nn.Sequential(nn.Linear(h + 3, h), nn.ReLU())
        self.recon = nn.Linear(h, NV)
        self.pred = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h))
        self.clf = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h, 1))
    def encode(self, X, st):
        out, hN = self.gru(X)
        return out, self.fuse(torch.cat([hN[-1], st], -1))
    def forward(self, X, st):
        return self.clf(self.encode(X, st)[1]).squeeze(-1)

def to_raw(z):
    y = z * _sd + _mu
    return torch.where(_islog, torch.expm1(y.clamp(max=12)), y)

def physiology_loss(pred_z, x_true_z, sel, m_in):
    raw_pred, raw_true = to_raw(pred_z), to_raw(torch.nan_to_num(x_true_z))
    C = torch.where(sel, raw_pred, raw_true)
    avail = sel | m_in
    g = lambda v: C[..., VIDX[v]]
    res = {
        "C1_MAP_formula": lambda: g("MAP") - (g("SBP") + 2 * g("DBP")) / 3,
        "C2_BP_ordering": lambda: torch.maximum(g("MAP") - g("SBP"), g("DBP") - g("MAP")),
        "C3_Henderson_Hasselbalch": lambda: g("pH") - (6.1 + torch.log10(g("HCO3").clamp(min=1e-2) / (0.03 * g("PaCO2").clamp(min=1e-2)))),
        "C4_Hct_3xHgb": lambda: g("Hct") - 3 * g("Hgb"),
        "C5_bilirubin_direct_le_total": lambda: g("Bilirubin_direct") - g("Bilirubin_total"),
        "C6_SaO2_vs_SpO2": lambda: g("SaO2") - g("O2Sat"),
    }
    losses = []
    for cid, c in CONS_P.items():
        J = [VIDX[v] for v in c["vars"]]
        ok = avail[..., J].all(-1) & sel[..., J].any(-1)
        if not ok.any():
            continue
        r = res[cid]()[ok]
        if c["kind"] == "eq":
            v = TF.relu((r - c["offset"]).abs() - c["tol"]) / c["scale"]
        else:
            v = TF.relu(r) / c["scale"]
        losses.append((v ** 2).clamp(max=100).mean())
    return torch.stack(losses).mean() if losses else pred_z.sum() * 0.0

CKPT = WORK / "checkpoints"; CKPT.mkdir(exist_ok=True)

def pretrain(arm, seed, tag=""):
    global INPUT_MODE
    mode, lp, lt, inp = VARIANTS[arm]
    INPUT_MODE = inp
    path = CKPT / f"pretrain_{arm}_{P}_s{seed}{tag}.pt"
    if path.exists():
        return torch.load(path, map_location=DEV)
    set_seed(seed)
    rng = np.random.default_rng(seed)
    net = Net(IN_DIM, CFG["hidden"]).to(DEV)
    opt = torch.optim.Adam(net.parameters(), lr=CFG["lr"])
    pool = IDX["train"]
    hist, t0 = [], time.time()
    for ep in range(CFG["pre_epochs"]):
        R_ep = rng.choice(pool, size=min(CFG["pre_windows_per_epoch"], len(pool)), replace=False)
        sums = {"rec": 0.0, "phys": 0.0, "temp": 0.0}; nb = 0
        net.train()
        for i in range(0, len(R_ep), CFG["batch"]):
            R = R_ep[i:i + CFG["batch"]]
            X, st, x, sel, m_in = build(R, mode, rng)
            if not sel.any():
                continue
            Xt, stt, xt, selt, mint = T(X), T(st), T(x), T(sel), T(m_in)
            out, z = net.encode(Xt, stt)
            pred = net.recon(out)
            rec = ((pred - torch.nan_to_num(xt)) ** 2)[selt].mean()
            loss = rec
            if lp > 0:
                phys = physiology_loss(pred, xt, selt, mint)
                loss = loss + lp * phys
                sums["phys"] += float(phys)
            if lt > 0:
                Rp = np.where(R > starts[row_pidx[R]], R - 1, R)
                Xp, stp, *_ = build(Rp)
                with torch.no_grad():
                    zp = net.encode(T(Xp), T(stp))[1]
                temp = (1 - TF.cosine_similarity(net.pred(z), zp, dim=-1)).mean()
                loss = loss + lt * temp
                sums["temp"] += float(temp)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            sums["rec"] += float(rec); nb += 1
        hist.append({"arm": arm, "protocol": P, "seed": seed, "epoch": ep, **{k: v / max(nb, 1) for k, v in sums.items()},
                     "seconds": time.time() - t0})
        log.info("pretrain %s %s s%d ep%d %s", arm, P, seed, ep, {k: round(v / max(nb, 1), 4) for k, v in sums.items()})
    state = net.state_dict()
    torch.save(state, path)
    pd.DataFrame(hist).to_csv(LOGS / f"pretrain_{arm}_{P}_s{seed}{tag}.csv", index=False)
    return state

def predict(net, R, bs=2048):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(R), bs):
            X, st, *_ = build(R[i:i + bs])
            out.append(torch.sigmoid(net(T(X), T(st))).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

def rows_of(split, frac, seed):
    col = f"{split}_s{seed}_f{int(round(frac * 100)):03d}"
    keep = subsets.index[subsets[col]].to_numpy()
    return IDX[split][np.isin(row_pidx[IDX[split]], keep)]

def train_neural(arm, frac, seed, init_state=None):
    global INPUT_MODE
    INPUT_MODE = VARIANTS[arm][3] if arm in VARIANTS else "full"
    set_seed(seed)
    rng = np.random.default_rng(1000 + seed)
    net = Net(IN_DIM, CFG["hidden"]).to(DEV)
    if init_state is not None:
        net.load_state_dict(init_state)
    opt = torch.optim.Adam(net.parameters(), lr=CFG["lr"])
    Rtr, Rva = rows_of("train", frac, seed), rows_of("val", frac, seed)
    if len(Rva) > CFG["val_rows_cap"]:
        Rva = np.sort(rng.choice(Rva, CFG["val_rows_cap"], replace=False))
    pw = torch.tensor(CFG["pos_weight"], device=DEV)
    best, best_state, bad, ep = -1.0, None, 0, 0
    t0 = time.time()
    for ep in range(CFG["ft_max_epochs"]):
        net.train()
        R_ep = rng.permutation(Rtr)[:CFG["ft_rows_per_epoch"]]
        for i in range(0, len(R_ep), CFG["batch"]):
            R = R_ep[i:i + CFG["batch"]]
            X, st, *_ = build(R)
            loss = TF.binary_cross_entropy_with_logits(net(T(X), T(st)), T(y_all[R]), pos_weight=pw)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        pv = predict(net, Rva)
        ap = average_precision_score(y_all[Rva], pv) if y_all[Rva].any() else 0.0
        if ap > best:
            best, bad = ap, 0
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
        if bad >= CFG["patience"]:
            break
    net.load_state_dict(best_state)
    return net, Rva, time.time() - t0, ep + 1

In [ ]:
import lightgbm as lgb
VITAL_IDX = [VIDX[v] for v in ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]]

def tab_features(R):
    R = np.asarray(R)
    p = row_pidx[R]; s0 = starts[p]
    lo = np.maximum(s0, R - 23)
    cnt24 = CS[R] - np.where((lo > 0)[:, None], CS[np.maximum(lo - 1, 0)], 0)
    idx6 = R[:, None] + np.arange(-5, 1)
    v6 = np.where((idx6 >= s0[:, None])[..., None], VN[np.maximum(idx6, 0)][:, :, VITAL_IDX], np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        roll = np.concatenate([np.nanmean(v6, 1), np.nanmin(v6, 1), np.nanmax(v6, 1), np.nanstd(v6, 1)], 1)
    return np.concatenate([FN[R], AGE[R], cnt24.astype(np.float32), roll,
                           np.stack([st_age[p], st_gen[p], row_icu[R]], 1)], 1).astype(np.float32)

def train_lgbm(frac, seed):
    Rtr, Rva = rows_of("train", frac, seed), rows_of("val", frac, seed)
    t0 = time.time()
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, min_child_samples=50, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(tab_features(Rtr), y_all[Rtr], eval_set=[(tab_features(Rva), y_all[Rva])], eval_metric="average_precision",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m, Rva, time.time() - t0

In [ ]:
import importlib.util
_spec = importlib.util.spec_from_file_location("official_eval", find_artifact("evaluate_sepsis_score_official.py", "01_dataset_acquisition"))
official = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(official)

def split_by_patient(R):
    p = row_pidx[R]
    cuts = np.flatnonzero(np.r_[True, p[1:] != p[:-1], True])
    return [(p[a], slice(a, b)) for a, b in zip(cuts[:-1], cuts[1:])]

_UCACHE = {}
def utility_norm(R, pred_bin):
    key = (len(R), int(R[0]), int(R[-1]))
    groups = split_by_patient(R)
    if key not in _UCACHE:
        best = inact = 0.0
        for pid, sl in groups:
            lab = y_all[R][sl]; n = len(lab)
            bp = np.zeros(n)
            if lab.any():
                ts = int(np.argmax(lab)) + 6
                bp[max(0, ts - 12):min(ts + 3 + 1, n)] = 1
            best += official.compute_prediction_utility(lab, bp)
            inact += official.compute_prediction_utility(lab, np.zeros(n))
        _UCACHE[key] = (best, inact)
    best, inact = _UCACHE[key]
    yR = y_all[R]
    obs = sum(official.compute_prediction_utility(yR[sl], pred_bin[sl]) for _, sl in groups)
    return (obs - inact) / (best - inact)

def choose_threshold(Rva, pva):
    qs = np.unique(np.quantile(pva, np.linspace(0.80, 0.995, 40)))
    scores = [utility_norm(Rva, (pva >= q).astype(int)) for q in qs]
    return float(qs[int(np.argmax(scores))])

def ece(y, p, bins=10):
    q = np.quantile(p, np.linspace(0, 1, bins + 1))
    b = np.clip(np.searchsorted(q, p, side="right") - 1, 0, bins - 1)
    tot = 0.0
    for k in range(bins):
        s = b == k
        if s.any():
            tot += s.mean() * abs(y[s].mean() - p[s].mean())
    return float(tot)

def evaluate(R, prob, thr):
    y = y_all[R]
    pb = (prob >= thr).astype(int)
    tp = int(((pb == 1) & (y == 1)).sum()); fp = int(((pb == 1) & (y == 0)).sum())
    fn = int(((pb == 0) & (y == 1)).sum()); tn = int(((pb == 0) & (y == 0)).sum())
    pdf = pd.DataFrame({"p": row_pidx[R], "prob": prob, "y": y, "alarm": pb, "t": row_t[R]})
    pat = pdf.groupby("p").agg(maxp=("prob", "max"), sep=("y", "max"))
    non = pdf[~pdf.p.isin(pat.index[pat.sep == 1])]
    leads = []
    tsep = patients["t_sepsis_idx"].to_numpy()
    for pid, g in pdf[pdf.p.isin(pat.index[pat.sep == 1])].groupby("p"):
        ts = tsep[pid]
        al = g.loc[(g.alarm == 1) & (g.t <= ts + 3), "t"]
        if len(al):
            leads.append(ts - al.min())
    sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); prec = tp / max(tp + fp, 1)
    return {"AUROC": roc_auc_score(y, prob), "AUPRC": average_precision_score(y, prob),
            "patient_AUROC": roc_auc_score(pat.sep, pat.maxp), "Brier": brier_score_loss(y, prob), "ECE": ece(y, prob),
            "utility": utility_norm(R, pb), "threshold": thr, "sensitivity": sens, "specificity": spec,
            "precision": prec, "F1": 2 * prec * sens / max(prec + sens, 1e-9),
            "false_alarms_per_patient_day": non.alarm.sum() / max(len(non) / 24, 1e-9),
            "septic_detected_pct": 100 * len(leads) / max((pat.sep == 1).sum(), 1),
            "median_lead_time_h": float(np.median(leads)) if leads else np.nan}

## Smoke test: every arm end-to-end on a few rows (fails fast on any bug)

In [ ]:
def run_one(arm, frac, seed, pre_state):
    if arm == "lightgbm":
        model, Rva, secs = train_lgbm(frac, seed)
        epochs = model.best_iteration_
        pva = model.predict_proba(tab_features(Rva))[:, 1]
        t1 = time.time(); pte = model.predict_proba(tab_features(IDX["test"]))[:, 1]; inf = time.time() - t1
    else:
        model, Rva, secs, epochs = train_neural(arm, frac, seed, pre_state)
        pva = predict(model, Rva)
        t1 = time.time(); pte = predict(model, IDX["test"]); inf = time.time() - t1
    thr = choose_threshold(Rva, pva)
    return pte, evaluate(IDX["test"], pte, thr), secs, inf, epochs

with StageTimer("smoke test", log):
    _saved = dict(CFG)
    CFG.update(pre_epochs=1, pre_windows_per_epoch=1024, ft_max_epochs=1, ft_rows_per_epoch=1024, val_rows_cap=1500)
    _IDXsave = IDX
    _rs = np.random.default_rng(0)
    IDX = {k: np.sort(_rs.choice(v, min(len(v), 3000), replace=False)) for k, v in _IDXsave.items()}
    for arm in MAIN_ARMS + ABL_ARMS:
        st_ = pretrain(arm, 99, tag="_smoke") if arm in VARIANTS else None
        _, met, *_ = run_one(arm, 0.10, 0, st_)
        log.info("smoke %s ok: AUROC=%.3f", arm, met["AUROC"])
    IDX = _IDXsave
    CFG.clear(); CFG.update(_saved)
    _UCACHE.clear()
    for f in CKPT.glob("*_smoke.pt"):
        f.unlink()
INPUT_MODE = "full"
log.info("SMOKE TEST PASSED")

## Run all jobs (resumable, time-guarded)

In [ ]:
RUNS = RES / "tables" / "runs_metrics.csv"
PRED_DIR = ART / "predictions"; PRED_DIR.mkdir(exist_ok=True)
def done_keys():
    if not RUNS.exists():
        return set()
    d = pd.read_csv(RUNS)
    return set(zip(d.protocol, d.arm, d.fraction.round(2), d.seed))
def pred_path(proto, arm, frac, seed):
    return PRED_DIR / f"pred_{arm}_{proto}_f{int(round(frac * 100)):03d}_s{seed}.npy"

skipped = []
for job in JOBS:
    if job["protocol"] != P:
        use_protocol(job["protocol"]); _UCACHE.clear()
    for seed in job["seeds"]:
        pre_state = {}
        for frac in job["fractions"]:
            for arm in job["arms"]:
                if (P, arm, round(frac, 2), seed) in done_keys():
                    continue
                if (time.time() - T_START) / 3600 > CFG["time_budget_h"]:
                    skipped.append((job["name"], P, arm, frac, seed)); continue
                if arm in VARIANTS and arm not in pre_state:
                    with StageTimer(f"pretrain {arm} {P} s{seed}", log):
                        pre_state[arm] = pretrain(arm, seed)
                log.info("=== %s | %s | %s | frac=%.2f | seed=%d | elapsed %.2f h ===", job["name"], P, arm, frac, seed,
                         (time.time() - T_START) / 3600)
                pte, met, secs, inf, epochs = run_one(arm, frac, seed, pre_state.get(arm))
                rec = {"job": job["name"], "protocol": P, "arm": arm, "fraction": frac, "seed": seed, "train_seconds": secs,
                       "test_inference_seconds": inf, "epochs_or_trees": epochs,
                       "n_train_patients": int(subsets[f"train_s{seed}_f{int(round(frac*100)):03d}"].sum()), **met}
                pd.DataFrame([rec]).to_csv(RUNS, mode="a", header=not RUNS.exists(), index=False)
                np.save(pred_path(P, arm, frac, seed), pte.astype(np.float32))
                print("RESULT", json.dumps({k: (round(v, 4) if isinstance(v, (float, np.floating)) else v) for k, v in rec.items()}, default=str), flush=True)
save_json({"skipped_runs": skipped}, RES / "skipped_runs.json")
log.info("All jobs finished; skipped=%d; elapsed %.2f h", len(skipped), (time.time() - T_START) / 3600)

## Multi-seed statistics

In [ ]:
runs = pd.read_csv(RUNS)
KEYM = ["AUROC", "AUPRC", "patient_AUROC", "utility", "Brier", "ECE", "sensitivity", "specificity",
        "false_alarms_per_patient_day", "septic_detected_pct", "median_lead_time_h"]
agg = runs.groupby(["job", "protocol", "arm", "fraction"])[KEYM].agg(["mean", "std", "count"])
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
agg = agg.reset_index()
agg.to_csv(RES / "tables" / "multiseed_summary.csv", index=False)

B = CFG["bootstrap_B"]
rng = np.random.default_rng(GLOBAL_SEED)
diff_rows, wins_rows, calib_rows = [], [], []
for proto in ["pooled", "A2B", "B2A"]:
    sub = runs[runs.protocol == proto]
    if sub.empty:
        continue
    use_protocol(proto)
    Rte = IDX["test"]; yte = y_all[Rte]
    sl = [s for _, s in split_by_patient(Rte)]
    BOOTS = []
    for _ in range(B):
        pick = rng.integers(0, len(sl), len(sl))
        BOOTS.append(np.concatenate([np.arange(sl[i].start, sl[i].stop) for i in pick]))
    for frac in sorted(sub.fraction.unique()):
        arms_here = sorted(sub[sub.fraction == frac].arm.unique())
        seeds_by_arm = {a: sorted(sub[(sub.fraction == frac) & (sub.arm == a)].seed) for a in arms_here}
        ens = {}
        for a in arms_here:
            ps = [np.load(pred_path(proto, a, frac, s)) for s in seeds_by_arm[a] if pred_path(proto, a, frac, s).exists()]
            if ps:
                ens[a] = np.mean(ps, axis=0)
        if "ptc_ssl" not in ens:
            continue
        bs = {a: np.array([[roc_auc_score(yte[b], pr[b]), average_precision_score(yte[b], pr[b])] for b in BOOTS]) for a, pr in ens.items()}
        for a in bs:
            if a == "ptc_ssl":
                continue
            d = bs["ptc_ssl"] - bs[a]
            for k, name in ((0, "AUROC"), (1, "AUPRC")):
                diff_rows.append({"protocol": proto, "fraction": frac, "comparison": f"ptc_ssl - {a}", "metric": name,
                                  "n_seeds_ptc": len(seeds_by_arm["ptc_ssl"]), "n_seeds_other": len(seeds_by_arm[a]),
                                  "mean_diff": d[:, k].mean(), "CI_low": np.percentile(d[:, k], 2.5), "CI_high": np.percentile(d[:, k], 97.5),
                                  "p_boot_two_sided": float(min(1, 2 * min((d[:, k] <= 0).mean(), (d[:, k] >= 0).mean())))})
            common = sorted(set(seeds_by_arm["ptc_ssl"]) & set(seeds_by_arm[a]))
            for name in ("AUROC", "AUPRC", "utility"):
                pv = sub[(sub.fraction == frac) & (sub.arm == "ptc_ssl") & sub.seed.isin(common)].set_index("seed")[name]
                ov = sub[(sub.fraction == frac) & (sub.arm == a) & sub.seed.isin(common)].set_index("seed")[name]
                dd = (pv - ov).dropna()
                wins_rows.append({"protocol": proto, "fraction": frac, "comparison": f"ptc_ssl - {a}", "metric": name,
                                  "seeds": len(dd), "ptc_wins": int((dd > 0).sum()), "mean_diff": dd.mean(), "sd_diff": dd.std()})
        for a, pr in ens.items():
            q = np.quantile(pr, np.linspace(0, 1, 11))
            b = np.clip(np.searchsorted(q, pr, side="right") - 1, 0, 9)
            for k in range(10):
                s = b == k
                if s.any():
                    calib_rows.append({"protocol": proto, "fraction": frac, "arm": a, "bin": k,
                                       "mean_pred": float(pr[s].mean()), "obs_rate": float(yte[s].mean()), "n": int(s.sum())})
        log.info("bootstrap done %s frac=%.2f", proto, frac)
diffs = pd.DataFrame(diff_rows); wins = pd.DataFrame(wins_rows); calib = pd.DataFrame(calib_rows)
diffs.to_csv(RES / "tables" / "paired_bootstrap_seed_ensemble.csv", index=False)
wins.to_csv(RES / "tables" / "per_seed_wins.csv", index=False)
calib.to_csv(RES / "tables" / "calibration_bins.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
main = agg[(agg.job == "main") & (agg.protocol == "pooled")]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, met in zip(axes, ["AUROC", "AUPRC", "utility"]):
    for arm, g in main.groupby("arm"):
        g = g.sort_values("fraction")
        ax.errorbar(g.fraction * 100, g[f"{met}_mean"], yerr=g[f"{met}_std"], marker="o", capsize=3, label=arm)
    ax.set_xscale("log"); ax.set_xlabel("% labelled training patients"); ax.set_title(f"{met} (mean ± SD over seeds)")
axes[0].legend(frameon=False, fontsize=7)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(RES / "figures" / f"fig_label_efficiency_multiseed.{ext}", dpi=300, bbox_inches="tight")
plt.close(fig)

## Compact printout (read by the paper draft) and result package

In [ ]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 500)
print("\n==== MULTISEED SUMMARY (mean | sd | n) ====")
for _, r in agg.iterrows():
    print("MS", r.job, r.protocol, r.arm, r.fraction, " ".join(f"{m}={r[m + '_mean']:.4f}|{r[m + '_std']:.4f}|{int(r[m + '_count'])}" for m in KEYM))
print("\n==== PAIRED BOOTSTRAP ON SEED ENSEMBLES ====")
for _, r in diffs.iterrows():
    print("PB", r.protocol, r.fraction, r.comparison, r.metric, f"{r.mean_diff:+.4f} [{r.CI_low:+.4f},{r.CI_high:+.4f}] p={r.p_boot_two_sided:.3f} seeds={r.n_seeds_ptc}/{r.n_seeds_other}")
print("\n==== PER-SEED WINS ====")
for _, r in wins.iterrows():
    print("WN", r.protocol, r.fraction, r.comparison, r.metric, f"wins={r.ptc_wins}/{r.seeds} mean={r.mean_diff:+.4f} sd={r.sd_diff:.4f}")
print("\nSKIPPED:", skipped)

import zipfile
with zipfile.ZipFile(WORK / "PTC_SSL_PAPER_RESULTS.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in RES.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(WORK))
    for f in LOGS.glob("*"):
        z.write(f, f.relative_to(WORK))
with zipfile.ZipFile(WORK / "PTC_SSL_IEEE_ACCESS_RESEARCH_PACKAGE.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in list(RES.rglob("*")) + list(LOGS.glob("*")) + list(CKPT.glob("*.pt")) + [ART / "ptcssl_data.py"]:
        if f.is_file():
            z.write(f, f.relative_to(WORK))
# keep the output small: drop large intermediate arrays (raw patient-level data is never packaged)
for f in ART.glob("X_*.npy"):
    f.unlink()
print("✅ Full experiments complete. Elapsed %.2f h" % ((time.time() - T_START) / 3600))